In [ ]:
def build_model_from_ckpt(ckpt, device):
    """
    Rebuild a VelocityNet model from a checkpoint.

    Expects checkpoint to store:
      - "latent_state_dim"
      - "model_state_dict"

    Optionally reads:
      - "hidden" (if you saved it; otherwise uses default)
      - input feature dims if present; otherwise assumes your standard layout:
            x = [z (D_state), e(1), I(1), SP(1), design(7)]
    """
    D_state    = int(ckpt["latent_state_dim"])
    state_dict = ckpt["model_state_dict"]

    # If you ever save these, we can use them; otherwise assume defaults
    hidden = int(ckpt.get("hidden", 128))
    dim_sp = int(ckpt.get("dim_sp", 1))
    dim_design = int(ckpt.get("dim_design", 7))
    dim_error = int(ckpt.get("dim_error", 1))
    dim_integral = int(ckpt.get("dim_integral", 1))

    model = VelocityNet(
        dim_state=D_state,
        dim_sp=dim_sp,
        dim_design=dim_design,
        dim_error=dim_error,
        dim_integral=dim_integral,
        hidden=hidden,
    ).to(device)

    model.load_state_dict(state_dict)
    model.eval()
    return model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_rollout_volume_and_error(roll, title_prefix=""):
    t        = roll["t"]
    v_true   = roll["vol_true"]
    v_pred   = roll["vol_pred"]
    # use precomputed error if present, else compute it
    err_vol  = roll.get("err_vol", np.abs(v_pred - v_true))

    r_modes  = roll.get("r_modes", None)
    sim_id   = roll.get("sim_id", None)

    # --- make a nice title ---
    pieces = []
    if title_prefix:
        pieces.append(title_prefix)
    if r_modes is not None:
        pieces.append(f"r={r_modes}")
    if sim_id is not None:
        pieces.append(f"sim={sim_id}")
    title_str = " | ".join(pieces) if pieces else "Rollout"

    fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

    # Top: volume trajectories
    ax0 = axes[0]
    ax0.plot(t, v_true, label="true volume")
    ax0.plot(t, v_pred, "--", label="pred volume")
    ax0.set_ylabel("Volume")
    ax0.set_title(title_str)
    ax0.legend()
    ax0.grid(alpha=0.3)

    # Bottom: absolute volume error
    ax1 = axes[1]
    ax1.plot(t, err_vol, label="|V_pred - V_true|")
    ax1.set_xlabel("Time [days]")
    ax1.set_ylabel("Abs volume error")
    ax1.grid(alpha=0.3)
    ax1.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# pick a validation sim
sim_test = int(aux["val_sims"][1])

# rollout
roll = rollout_single_sim(r_modes=r, sim_id=sim_test, max_steps=None)
ckpt = torch.load(f"models/latent_velocity_model_r{r}.pt", map_location="cpu")

train_losses = ckpt["train_losses"]
val_losses   = ckpt["val_losses"]

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title(f"Training / validation loss (r={r})")
plt.legend()
plt.grid(True)
plt.show()
# your existing plotting utilities should still work if they only rely on volume / z
plot_rollout_volume_and_error(roll, title_prefix="NODE (state-only)")


In [ ]:
# --- Displacement POD + snapshot path info (adjust names if needed) ---
DISP_POD_GROUP       = "pod_full"     # inside displacements.zarr
DISP_POD_U_NAME      = "U"            # dataset with basis vectors
DISP_POD_MEAN_NAME   = "mean"         # dataset with mean
DISP_SNAP_DATASET    = "snapshots"    # full-order snapshots in displacements.zarr


In [ ]:
import numpy as np
import zarr

def get_growth_params_for_sim(sim_id: int):
    """
    Map sim_id -> job_id using displacements.zarr/mapping/job_index_per_sim,
    then read the correct row from the design file:
        'PGOH_Ideal_tol_Tcrit_V_mu_kk1_kk2_kappa_k1_design.txt'

    The design file:
        - Has a header row to skip
        - Each subsequent row corresponds to job_id = line_number (1-based)
        - Columns are comma- or space-delimited
        - We extract columns 2, 5, 6 (0-based) as:
             theta_crit = col 2
             k1         = col 5
             k2         = col 6
    """

    # --- 1) sim_id -> job_id ---
    g_disp = zarr.open("displacements.zarr", mode="r")
    job_index_per_sim = g_disp["mapping"]["job_index_per_sim"][:]  # shape (n_sims,)
    job_id = int(job_index_per_sim[sim_id])   # job index for this sim (1-based!)

    # --- 2) Load design table from text file ---
    design_file = "PGOH_Ideal_tol_Tcrit_V_mu_kk1_kk2_kappa_k1_design.txt"

    # Load skipping header, allowing space or comma separators
    design_table = np.loadtxt(design_file, skiprows=0)

    # Because job_id is 1-based, convert to 0-based row index:
    row_index = job_id - 1

    if row_index < 0 or row_index >= design_table.shape[0]:
        raise IndexError(
            f"Job ID {job_id} (row idx {row_index}) is out of range for design table "
            f"with {design_table.shape[0]} rows."
        )

    row = design_table[row_index]

    # --- 3) Extract θ_crit, k1, k2 ---
    theta_crit = float(row[2])   # column 2 (0-based index)
    k1         = float(row[5])   # column 5
    k2         = float(row[6])   # column 6

    return job_id, theta_crit, k1, k2


In [ ]:
g_design = zarr.open("Sim_design_params_7_dims.zarr", mode="r")
design_snap = g_design["design_params_per_snapshot"][:]  # (n_jobs, n_design_dims)
print(design_snap[1])

In [ ]:
def rollout_single_sim(
    r_modes: int,
    sim_id: int = 0,
    max_steps: int | None = None,
):
    """
    Forward-Euler rollout for one simulation with the *state-only* NODE.

    State NODE:  z = [u_latents (r), volume (1)]  (dim_state = r+1)

    Args
    ----
    r_modes : int
        Number of POD modes (latent size for u).
    sim_id : int
        Simulation index into the raw dataset.
    max_steps : int | None
        Max rollout steps; if None, use all T_true - 1 available for this sim.

    Returns
    -------
    rollout : dict
        {
          "t"         : (T_roll,),
          "z_true"    : (T_roll, r_modes+1),
          "z_pred"    : (T_roll, r_modes+1),
          "vol_true"  : (T_roll,),
          "vol_pred"  : (T_roll,),
          "err_L2"    : (T_roll,),
          "err_vol"   : (T_roll,),
          "e_hist"    : (T_roll,),
          "I_hist"    : (T_roll,),
          "r_modes"   : int,
          "sim_id"    : int,
          "volume_dim": int,
        }
    """
    # ----------------------------------------------------
    # 1) Load NODE checkpoint
    # ----------------------------------------------------
    ckpt_path = f"models/latent_velocity_model_r{r_modes}.pt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    D_state     = ckpt["latent_state_dim"]          # r + 1
    r_ckpt      = ckpt["r_modes"]
    dt          = ckpt["dt"]
    z_mean      = ckpt["z_mean"]
    z_std       = ckpt["z_std"]
    sp_mean     = ckpt["sp_mean"]
    sp_std      = ckpt["sp_std"]
    design_mean = ckpt["design_mean"]
    design_std  = ckpt["design_std"]
    e_mean      = ckpt["e_mean"]
    e_std       = ckpt["e_std"]
    I_mean      = ckpt["I_mean"]
    I_std       = ckpt["I_std"]

    assert r_ckpt == r_modes, f"Checkpoint r={r_ckpt} != requested r={r_modes}"
    volume_idx = r_modes   # last entry in z

    # rebuild model from ckpt
    model = VelocityNetWithGrowth(
        dim_state=D_state,
        dim_input=ckpt["input_dim"],
        r_modes=r_modes
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # ----------------------------------------------------
    # 2) Load raw latent/volume/design data
    # ----------------------------------------------------
    (U_lat, Lx_lat, Ly_lat,
     volume_snap, sim_index, time_vals,
     volume_SP, design_all,
     dt_data, n_sims) = load_raw_data(r_modes)

    # state for NODE: [u, volume] (raw)
    Z_state_all = np.concatenate(
        [U_lat, volume_snap.reshape(-1, 1)],
        axis=1
    ).astype(np.float32)  # (S, r+1)

    # select frames for this sim
    idx = np.where(sim_index == sim_id)[0]
    if idx.size < 2:
        raise ValueError(f"Sim {sim_id} has too few frames ({idx.size}).")

    t_s   = time_vals[idx]
    order = np.argsort(t_s)
    idx_sorted = idx[order]

    Z_true_sim = Z_state_all[idx_sorted, :]    # (T_true, r+1)
    SP_sim     = volume_SP[idx_sorted]         # (T_true,)
    design_sim = design_all[idx_sorted, :]     # (T_true, 7)
    t_sim      = t_s[order]                    # (T_true,)

    # ---- DEBUG: print true simulation length ----
    T_true = Z_true_sim.shape[0]
    dt_list = np.diff(t_sim)

    print(f"[sim {sim_id}] --- FE SIM INFO ---")
    print(f"  T_true (num FE frames): {T_true}")
    print(f"  t_sim[0] = {t_sim[0]:.4f} days")
    print(f"  t_sim[-1] = {t_sim[-1]:.4f} days")
    print(f"  mean FE Δt = {dt_list.mean():.6f} days")
    print(f"  min FE Δt  = {dt_list.min():.6f} days")
    print(f"  max FE Δt  = {dt_list.max():.6f} days")
    
    T_true = Z_true_sim.shape[0]
    if max_steps is None:
        n_steps = T_true - 1
    else:
        n_steps = min(max_steps, T_true - 1)
    T_roll = n_steps + 1

    # design is constant per sim
    design_raw  = design_sim[0:1, :]                       # (1,7)
    design_norm = (design_raw - design_mean) / design_std
    design_tensor = torch.from_numpy(design_norm.astype(np.float32)).to(device)

    # ----------------------------------------------------
    # 3) Initial NODE state and PI variables
    # ----------------------------------------------------
    Z0_raw  = Z_true_sim[0, :]           # (r+1,)
    V0_raw  = Z0_raw[volume_idx]
    SP0_raw = SP_sim[0]
    e_raw   = SP0_raw - V0_raw
    I_raw   = 0.0

    z0_norm = ((Z0_raw - z_mean[0]) / z_std[0]).astype(np.float32).reshape(1, -1)
    z_curr  = torch.from_numpy(z0_norm).to(device)

    # histories
    z_true_list   = [Z0_raw.copy()]
    z_pred_list   = [Z0_raw.copy()]
    t_roll_list   = [t_sim[0]]
    vol_true_list = [V0_raw]
    vol_pred_list = [V0_raw]
    e_hist        = [e_raw]
    I_hist        = [I_raw]

    # ----------------------------------------------------
    # 4) Rollout (NODE only, no growth)
    # ----------------------------------------------------
    with torch.no_grad():
        for k in range(n_steps):
            t_k  = t_sim[k]
            t_k1 = t_sim[k+1]
            dt_k = t_k1 - t_k   # physical timestep (~1.4 days)

            SP_k_raw = SP_sim[k]

            # normalize e, I, SP
            e_norm  = (e_raw - e_mean[0]) / e_std[0]
            I_norm  = (I_raw - I_mean[0]) / I_std[0]
            sp_norm = (SP_k_raw - sp_mean[0]) / sp_std[0]

            e_tensor  = torch.tensor([[e_norm]], dtype=torch.float32, device=device)
            I_tensor  = torch.tensor([[I_norm]], dtype=torch.float32, device=device)
            sp_tensor = torch.tensor([[sp_norm]], dtype=torch.float32, device=device)

            x_k = torch.cat([z_curr, e_tensor, I_tensor, sp_tensor, design_tensor], dim=1)

            dz_norm, _, _ = model(x_k)   # ignore growth outputs for NODE

            # ----- NODE state update for [u_latent, V] -----
            z_next    = z_curr + dt * dz_norm          # dt from training
            z_next_np = z_next.cpu().numpy()[0]
            Z_next_raw = z_next_np * z_std[0] + z_mean[0]   # back to physical

            V_next_raw = Z_next_raw[volume_idx]
            V_true_raw = Z_true_sim[k+1, volume_idx]

            # store NODE rollout
            z_pred_list.append(Z_next_raw)
            z_true_list.append(Z_true_sim[k+1, :])
            t_roll_list.append(t_k1)
            vol_pred_list.append(V_next_raw)
            vol_true_list.append(V_true_raw)

            # ----- PI update (volume control) -----
            I_raw       = I_raw + dt_k * e_raw
            SP_next_raw = SP_sim[k+1]
            e_raw       = SP_next_raw - V_next_raw
            e_hist.append(e_raw)
            I_hist.append(I_raw)

            z_curr = z_next

    # ----------------------------------------------------
    # 5) Pack outputs
    # ----------------------------------------------------
    z_true_raw = np.vstack(z_true_list)
    z_pred_raw = np.vstack(z_pred_list)
    t_roll_arr = np.array(t_roll_list)
    vol_true   = np.array(vol_true_list)
    vol_pred   = np.array(vol_pred_list)

    err_L2  = np.linalg.norm(z_pred_raw - z_true_raw, axis=1)
    err_vol = np.abs(vol_pred - vol_true)

    rollout = {
        "t": t_roll_arr,
        "z_true": z_true_raw,
        "z_pred": z_pred_raw,
        "vol_true": vol_true,
        "vol_pred": vol_pred,
        "err_L2": err_L2,
        "err_vol": err_vol,
        "r_modes": r_modes,
        "sim_id": sim_id,
        "volume_dim": volume_idx,
        "e_hist": np.array(e_hist),
        "I_hist": np.array(I_hist),
    }

    print(f"[rollout] r={r_modes}, sim={sim_id}, steps={n_steps}, dt={dt:.4f} days")
    print("  Final L2 error:", err_L2[-1])
    print("  Final volume error:", err_vol[-1])

    return rollout


In [ ]:
import numpy as np

def get_U_at_times(t_all, U_all, times_to_plot, surface_idx=None):
    """
    t_all        : (Nt,) array of times from the displacement results
    U_all        : (Nt, N_nodes_full, 3) or (Nt, N_nodes_surface, 3)
    times_to_plot: list/array of physical times you want to visualize
    surface_idx  : optional 1D integer array; if U_all is full mesh, this
                   indexes the surface nodes.

    Returns:
        U_slices: list of (N_surface_nodes, 3) arrays
        idx_list: list of integer indices into t_all
    """
    t_all = np.asarray(t_all)
    times_to_plot = np.asarray(times_to_plot)
    
    U_slices = []
    idx_list = []

    for t_query in times_to_plot:
        # nearest time index
        idx = int(np.argmin(np.abs(t_all - t_query)))
        idx_list.append(idx)

        U_full_step = U_all[idx]  # (N_nodes_full, 3) or (N_nodes_surface, 3)
        if surface_idx is not None:
            U_step = U_full_step[surface_idx, :]   # (N_surface_nodes, 3)
        else:
            U_step = U_full_step                   # already surface

        U_slices.append(U_step)

    return U_slices, idx_list


In [ ]:
def reconstruct_u_nodes_sequence_from_rollout(roll, r_modes):
    """
    Decode NODE displacements from rollout latent sequence.
    MUST match flattening used in training: [ux_block, uy_block, uz_block].
    """

    z_pred = roll["z_pred"]   # (T, D_state)
    T = z_pred.shape[0]

    # load POD basis + mean
    g_disp = zarr.open_group(ZARR_DISP, mode="r")
    U_full = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_U_NAME}"])
    mu     = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_MEAN_NAME}"])

    U_r = U_full[:, :r_modes]     # (M, r)
    M = U_r.shape[0]
    N = M // 3                    # number of nodes

    u_nodes_seq = np.zeros((T, N, 3), dtype=float)

    for k in range(T):
        # extract latent displacement coeffs
        alpha = z_pred[k, :r_modes]

        # decode flat displacement
        u_flat = mu + U_r @ alpha    # (M,)

        # --- CRITICAL FIX: apply block-split ordering ---
        ux = u_flat[0:N]
        uy = u_flat[N:2*N]
        uz = u_flat[2*N:3*N]

        u_nodes_seq[k] = np.stack([ux, uy, uz], axis=1)  # (N,3)

    return roll["t"], u_nodes_seq


In [ ]:
import numpy as np
import pandas as pd

# -----------------------------------------------------
# LOAD NODES (11163 nodes, 3 layers of 3721 nodes each)
# -----------------------------------------------------
node_df   = pd.read_csv("GOH_Nodes_Test_for_Visualization.csv", header=None)
node_xyz  = node_df.iloc[:, 1:4].to_numpy()       # (11163,3)
N_total   = node_xyz.shape[0]
N_layer   = N_total // 3                          # 3721

# Keep only the bottom layer nodes
bottom_global_start = 2 * N_layer                 # index 7442
bottom_global_end   = 3 * N_layer                 # index 11163
bottom_node_global_idx = np.arange(bottom_global_start, bottom_global_end)

node_coords_bottom = node_xyz[bottom_node_global_idx]   # (3721,3)
N_bottom = node_coords_bottom.shape[0]

print("Total nodes:      ", N_total)
print("Nodes per layer:  ", N_layer)
print("Bottom nodes:     ", N_bottom)


# -----------------------------------------------------
# LOAD ELEMENT CONNECTIVITY (7200 hex elements)
# -----------------------------------------------------
elem_df = pd.read_csv("GOH_Elements_Test_for_Visualization.csv", header=None)
elem_conn_full = elem_df.iloc[:, 1:].to_numpy(int)   # minus label col
elem_conn_full -= 1                                   # convert 1-based → 0-based

N_elem_total = elem_conn_full.shape[0]
N_elem_layer = N_elem_total // 2                      # 3600

# Keep only the bottom-layer elements
bottom_elem_start = N_elem_layer                      # 3600
bottom_elem_end   = N_elem_total                      # 7200

elem_conn_bottom_global = elem_conn_full[bottom_elem_start : bottom_elem_end]   # (3600,8)

print("Total elements:   ", N_elem_total)
print("Bottom elements:  ", elem_conn_bottom_global.shape[0])


# -----------------------------------------------------
# EXTRACT BOTTOM FACE CONNECTIVITY FOR THESE ELEMENTS
# -----------------------------------------------------
# For each bottom-layer hex element, the *bottom face* nodes are the last 4 entries:
#   [ n4, n5, n6, n7 ]  = elem_conn_bottom_global[:, 4:8]

bottom_face_conn_global = elem_conn_bottom_global[:, 4:8]   # shape = (3600,4)

print("bottom_face_conn_global shape:", bottom_face_conn_global.shape)


# -----------------------------------------------------
# CONVERT GLOBAL NODE INDICES → LOCAL BOTTOM-NODE INDICES
# -----------------------------------------------------
# bottom layer nodes occupy global indices 7442..11162
# so local_index = global_index - 2*N_layer = global_index - 7442

bottom_face_conn_local = bottom_face_conn_global - bottom_global_start

print("Local bottom face shape:", bottom_face_conn_local.shape)
print("Local index min/max:", bottom_face_conn_local.min(), bottom_face_conn_local.max())
assert bottom_face_conn_local.min() >= 0
assert bottom_face_conn_local.max() < N_bottom


In [ ]:
node_coords_bottom[:5]


In [ ]:
import numpy as np

def shape_function_gradients(xi, eta):
    """Return arrays dN_dxi, dN_deta of length 4 for bilinear quad."""
    dN_dxi = np.array([
        -(1 - eta) / 4,
         (1 - eta) / 4,
         (1 + eta) / 4,
        -(1 + eta) / 4
    ])
    dN_deta = np.array([
        -(1 - xi) / 4,
        -(1 + xi) / 4,
         (1 + xi) / 4,
         (1 - xi) / 4
    ])
    return dN_dxi, dN_deta

# Example: compute Gξ and Gη for one element
# Suppose nodal coordinates X are shape (4,3), each row is a node [x,y,z]
X_elem = np.array([
    [0.0, 0.0, 0.0],   # node 1
    [1.0, 0.0, 0.0],   # node 2
    [1.0, 1.0, 0.0],   # node 3
    [0.0, 1.0, 0.0]    # node 4
])

xi, eta = 0.0, 0.0  # evaluate at element center
dN_dxi, dN_deta = shape_function_gradients(xi, eta)

# Gξ and Gη are 3D vectors
Gxi = X_elem.T @ dN_dxi   # (3,) vector
Geta = X_elem.T @ dN_deta # (3,) vector

print("Gxi =", Gxi)
print("Geta =", Geta)


In [ ]:
def evalN1(xi,eta):
  N1 = (1 - xi)*(1 - eta) / 4
  return N1
def evalN2(xi,eta):
  N2 = (1 + xi)*(1 - eta) / 4
  return N2
def evalN3(xi,eta):
  N3 = (1 + xi)*(1 + eta) / 4
  return N3
def evalN4(xi,eta):
  N4 = (1 - xi)*(1 + eta) / 4
  return N4
def dN1_dxi(xi, eta):
    return -(1 - eta) / 4
def dN1_deta(xi, eta):
    return -(1 - xi) / 4

def dN2_dxi(xi, eta):
    return (1 - eta) / 4
def dN2_deta(xi, eta):
    return -(1 + xi) / 4

def dN3_dxi(xi, eta):
    return (1 + eta) / 4
def dN3_deta(xi, eta):
    return (1 + xi) / 4

def dN4_dxi(xi, eta):
    return -(1 + eta) / 4
def dN4_deta(xi, eta):
    return (1 - xi) / 4


In [ ]:
# same as before but in a function
def integrate_growth(node_coords, elem_conn, node_u, elem_lamdag, k1, k2, theta_crit, time, dt = 1.4):
  ## Integrate growth
  # Input:
  #   initial nodal coordinates in node_coords
  #   displacement nodal values at time=t stored in node_u
  #   growth values at time=t stored at the element level element_lamdag
  #   principal directions for growth defined in reference configuration (unchanged throughout simulation)
  #   parameters
  # Output:
  #   updated growth values at time t+delta_t stored at the element level
  n_nodes = node_coords.shape[0]
  n_elem = elem_conn.shape[0]

  ## Loop over elements
  for ei in range(n_elem):
    node_id_ei = elem_conn[ei]
    X_ni = node_coords[node_id_ei]
    u_ni = node_u[node_id_ei]
    x_ni = X_ni + u_ni

    # let's eval at center of element
    xi=0
    eta=0
    dN_dxi, dN_deta = shape_function_gradients(xi, eta)

    # Gξ and Gη are 3D vectors
    Gxi = X_ni.T @ dN_dxi   # (3,) vector
    Geta = X_ni.T @ dN_deta # (3,) vector
    # Normal
    N = np.cross(Gxi,Geta)/np.linalg.norm(np.cross(Gxi,Geta))

    # invert the Jacobian
    G = np.column_stack((Gxi, Geta, N))
    G_inv = np.linalg.inv(G)

    G_dual_xi  = G_inv[0, :]   # row 0
    G_dual_eta = G_inv[1, :]   # row 1
    G_dual_N   = G_inv[2, :]   # row 2

    # build the covariant basis for the deformed surface
    gxi = x_ni.T @ dN_dxi   # (3,) vector
    geta = x_ni.T @ dN_deta # (3,) vector
    n = np.cross(gxi,geta)/np.linalg.norm(np.cross(gxi,geta))

    # total deformation gradient assuming plane strain
    F = np.outer(gxi,G_dual_xi) + np.outer(geta,G_dual_eta) + np.outer(n,N)

    # Growth tensor from the previous time point
    lamda1g_t = elem_lamdag[ei,0]
    lamda2g_t = elem_lamdag[ei,1]
    # unit vectors for growth are defined in the original configuration
    # note, these could in principle change over the geometry but for our case they are
    # always the 1,0,0 and 0,1,0 vectors because the initial geometry is planar and
    # we assumed the growth axis aligned with the main X and Y coord axis
    a0 = np.array([1,0,0])
    s0 = np.array([0,1,0])
    # note, no growth in normal direction
    Fg_t = lamda1g_t*np.outer(a0,a0)+lamda2g_t*np.outer(s0,s0)+np.outer(N,N)

    # given previous growth and new total deformation, solve for new growth value
    # forward Euler or Backward Euler?
    # for ease, let's do forward Euler, but backward is not too difficult
    # F = Fe Fg, Fe = F Fginv
    Fg_inv_t = np.linalg.inv(Fg_t)
    Fe_t = np.dot(F,Fg_inv_t)
    Ce_t = np.dot(Fe_t.T,Fe_t)
    # Get corresponding elastic stretches for a0 and s0 directions
    lam1e_t = np.sqrt(np.dot(a0,np.dot(Ce_t,a0)))
    lam2e_t = np.sqrt(np.dot(s0,np.dot(Ce_t,s0)))
    # Integrate growth
    if lam1e_t>=theta_crit:
      lam1gdot = k1*(lam1e_t-theta_crit)
    else:
      lam1gdot = 0 # no negative growth allowed I guess
    if lam2e_t>=theta_crit:
      lam2gdot = k2*(lam2e_t-theta_crit)
    else:
      lam2gdot = 0
    if 0 < time < 7:
        lam1gdot = 0
        lam2gdot = 0
    if 14 < time < 21:
        lam1gdot = 0
        lam2gdot = 0
    if 28 < time < 35:
        lam1gdot = 0
        lam2gdot = 0
    if 42 < time < 49:
        lam1gdot = 0
        lam2gdot = 0
    if 56 < time < 63:
        lam1gdot = 0
        lam2gdot = 0
    if 70 < time < 77:
        lam1gdot = 0
        lam2gdot = 0
    if 84 < time < 91:
        lam1gdot = 0
        lam2gdot = 0
    if 98 < time < 105:
        lam1gdot = 0
        lam2gdot = 0
    lamda1g = lamda1g_t + dt*lam1gdot
    lamda2g = lamda2g_t + dt*lam2gdot

    # update the element array directly. Although one could just output the whole array
    # and do the replacement outside this function
    elem_lamdag[ei,0] = lamda1g
    elem_lamdag[ei,1] = lamda2g
  return elem_lamdag

In [ ]:
import zarr
import numpy as np

# 1) Run NODE rollout (no growth inside)
r_modes = 15
sim_id  = aux["val_sims"][0]

# NODE rollout (no growth inside)
roll = rollout_single_sim(
    r_modes=r_modes,
    sim_id=sim_id,
    max_steps=None,
)

# Reconstruct nodal displacements with the fixed decoder
t, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
    roll=roll,
    r_modes=r_modes,
)

# Final time-step nodal displacement
u_final_nodes = u_nodes_seq[-1]          # (N_nodes, 3)
uz_from_seq   = u_final_nodes[:, 2]      # (N_nodes,)
import matplotlib.pyplot as plt
import numpy as np

# node_coords_bottom is (3721, 3) → take x,y only
x = node_coords_bottom[:, 0]
y = node_coords_bottom[:, 1]

plt.figure(figsize=(6,5))
sc = plt.scatter(x, y, c=uz_from_seq, s=8, cmap="viridis")

plt.gca().set_aspect("equal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("uz_from_seq at final time")
plt.xlim(0, 300)
plt.ylim(0, 300)

cbar = plt.colorbar(sc)
cbar.set_label("uz displacement (mm)")

plt.tight_layout()
plt.show()

# For plotting, use your bottom_xy (must correspond to the same nodes as POD)
bottom_xy = node_coords_bottom[:, :2]


T = len(t)
N_elem = bottom_face_conn_local.shape[0]

# 3) Get job-specific growth parameters for this sim
job_id, theta_crit, k1, k2 = get_growth_params_for_sim(sim_id)
print(f"Sim {sim_id} -> job {job_id} with theta_crit={theta_crit:.4f}, k1={k1:.4f}, k2={k2:.4f}")

# 4) Initialize predicted growth field on bottom faces (all ones)
elem_lamdag = np.ones((N_elem, 2), dtype=float)
lamdag_hist_pred = np.zeros((T, N_elem, 2))
lamdag_hist_pred[0] = elem_lamdag.copy()

# 5) Integrate predicted growth with job-specific (k1, k2, theta_crit)
for k in range(T - 1):
    dt_k = t[k+1] - t[k]
    node_u_k = u_nodes_seq[k]  # (N_nodes_bottom,3)

    elem_lamdag = integrate_growth(
        node_coords=node_coords_bottom,
        elem_conn=bottom_face_conn_local,
        node_u=node_u_k,
        elem_lamdag=elem_lamdag,
        k1=k1,
        k2=k2,
        theta_crit=theta_crit,
        time=t[k],
        dt=dt_k,
    )
    lamdag_hist_pred[k+1] = elem_lamdag.copy()

# Save predicted growth into rollout
roll["lamdag_hist_pred"] = lamdag_hist_pred
roll["job_id"]      = job_id
roll["theta_crit"]  = theta_crit
roll["k1"]          = k1
roll["k2"]          = k2


In [ ]:
print("node_coords_bottom.shape:", node_coords_bottom.shape)
print("bottom_face_conn_local.shape:", bottom_face_conn_local.shape)
print("max(bottom_face_conn_local):", bottom_face_conn_local.max())
print("min(bottom_face_conn_local):", bottom_face_conn_local.min())

N_bottom = node_coords_bottom.shape[0]
assert bottom_face_conn_local.max() < N_bottom, (
    f"Connectivity uses index {bottom_face_conn_local.max()} "
    f"but node_coords_bottom has only {N_bottom} nodes"
)


In [ ]:
import zarr
import numpy as np

# --- 1) Get snapshot indices for this sim, same as rollout alignment ---

(
    U_lat, Lx_lat, Ly_lat,
    volume_snap, sim_index, time_vals,
    volume_SP, design_all,
    dt_data, n_sims
) = load_raw_data(r_modes)

snap_idx = np.where(sim_index == sim_id)[0]
if snap_idx.size < 1:
    raise ValueError(f"No snapshots found for sim_id={sim_id}")

t_s      = time_vals[snap_idx]
order    = np.argsort(t_s)
idx_sorted = snap_idx[order]       # (T,) global snapshot indices

T = idx_sorted.shape[0]
print("T (rollout length) =", T)

# --- 2) Grab job_index_per_snapshot to infer snapshot axis ---

g_disp = zarr.open("displacements.zarr", mode="r")
job_index_per_snapshot = g_disp["mapping"]["job_index_per_snapshot"][:]  # (S_total,)
S_total = job_index_per_snapshot.shape[0]

# --- 3) Load SDV snapshots from ip_nodal.zarr ---

g_ip = zarr.open("ip_nodal.zarr", mode="r")
lamgx_all = g_ip["snaps_SDV1_bottom"][:]  # SDV1 (λ_gx)
lamgy_all = g_ip["snaps_SDV4_bottom"][:]  # SDV4 (λ_gy)

print("lamgx_all shape:", lamgx_all.shape)
print("lamgy_all shape:", lamgy_all.shape)
print("S_total snapshots:", S_total)

# We expect either:
#   (S_total, N_bottom) or (N_bottom, S_total)
# Detect which axis is snapshots:
if lamgx_all.shape[0] == S_total:
    # axis 0 = snapshots, axis 1 = nodes
    # select snapshots for this sim along axis 0
    lamgx_sim_snap_node = lamgx_all[idx_sorted, :]   # (T, N_bottom)
    lamgy_sim_snap_node = lamgy_all[idx_sorted, :]   # (T, N_bottom)

elif lamgx_all.shape[1] == S_total:
    # axis 1 = snapshots, axis 0 = nodes
    # select snapshots for this sim along axis 1, then transpose
    lamgx_sim_snap_node = lamgx_all[:, idx_sorted].T  # (T, N_bottom)
    lamgy_sim_snap_node = lamgy_all[:, idx_sorted].T  # (T, N_bottom)

else:
    raise ValueError(
        f"Cannot infer snapshot axis. lamgx_all shape={lamgx_all.shape}, S_total={S_total}"
    )

# optional check: number of bottom nodes must match
N_bottom = node_coords_bottom.shape[0]
assert lamgx_sim_snap_node.shape == (T, N_bottom), \
    f"Unexpected λ_gx shape {lamgx_sim_snap_node.shape}, expected ({T},{N_bottom})"

assert lamgy_sim_snap_node.shape == (T, N_bottom), \
    f"Unexpected λ_gy shape {lamgy_sim_snap_node.shape}, expected ({T},{N_bottom})"

# optional: consistency with job_id
# note: job_index_per_snapshot is (S_total,), so index with idx_sorted
assert np.all(job_index_per_snapshot[idx_sorted] == job_id), \
    "Mismatch between job_id from job_index_per_sim and job_index_per_snapshot."

# --- 4) Pack into lamdag_hist_true: (T, N_bottom, 2) ---

lamdag_hist_true = np.stack(
    [lamgx_sim_snap_node, lamgy_sim_snap_node],
    axis=-1
)  # (T, N_bottom, 2)

print("lamdag_hist_true shape:", lamdag_hist_true.shape)

# attach to rollout
roll["lamdag_hist_true"] = lamdag_hist_true


In [ ]:
# --- Displacement POD + snapshot path info (adjust names if needed) ---
DISP_POD_GROUP       = "pod_full"     # inside displacements.zarr
DISP_POD_U_NAME      = "U"            # dataset with basis vectors
DISP_POD_MEAN_NAME   = "mean"         # dataset with mean
DISP_SNAP_DATASET    = "snapshots"    # full-order snapshots in displacements.zarr


In [ ]:
def reconstruct_displacement_errors(r_modes: int,
                                    sim_id: int = 0,
                                    max_steps: int | None = None):
    """
    For a given (r_modes, sim_id):
        1) Run latent rollout (using rollout_single_sim).
        2) For each time step, decode displacement from latent coeffs using POD basis.
        3) Compare to the true displacement snapshot from displacements.zarr.
    
    Returns:
        dict with:
          - t:             (T,) time [days]
          - err_L2:        (T,) ||u_pred - u_true||_2
          - err_rel_L2:    (T,) ||u_pred - u_true||_2 / ||u_true||_2
          - u_pred_sample: (M,) displacement at final time (predicted)
          - u_true_sample: (M,) displacement at final time (true)
    """
    # ---- 1) latent rollout ----
    rollout = rollout_single_sim(r_modes=r_modes, sim_id=sim_id, max_steps=max_steps)
    z_pred   = rollout["z_pred"]      # (T, D_state)   latent+volume (raw)
    t_roll   = rollout["t"]           # (T,)
    T, D_state = z_pred.shape
    r = r_modes

    # ---- 2) mapping from sim frames to global snapshot indices ----
    #      (same logic as in rollout, but we just need idx_sorted)
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, n_sims) = load_raw_data(r_modes)

    idx = np.where(sim_index == sim_id)[0]
    if idx.size < 2:
        raise ValueError(f"Sim {sim_id} has too few frames ({idx.size}).")

    t_s = time_vals[idx]
    order = np.argsort(t_s)
    idx_sorted = idx[order]

    # Make sure we only use as many snapshots as in the rollout
    if T > idx_sorted.size:
        raise ValueError(f"Rollout length T={T} > #frames for sim {sim_id}={idx_sorted.size}")
    snap_indices = idx_sorted[:T]   # global snapshot index for each rollout time

    # ---- 3) load POD basis & mean for displacement, and snapshot matrix ----
    g_disp = zarr.open_group(ZARR_DISP, mode="r")

    U_full = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_U_NAME}"], dtype=np.float32)  # (M, r_full)
    mu     = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_MEAN_NAME}"], dtype=np.float32)  # (M,)
    U_r    = U_full[:, :r]   # (M, r)

    disp_snaps = g_disp[DISP_SNAP_DATASET]   # zarr array (M, S)

    M = U_r.shape[0]

    err_L2     = np.zeros(T, dtype=np.float64)
    err_rel_L2 = np.zeros(T, dtype=np.float64)

    # ---- 4) loop over time steps: reconstruct & compare ----
    for k in range(T):
        snap_idx = int(snap_indices[k])

        # true displacement snapshot (full-order)
        u_true = np.asarray(disp_snaps[:, snap_idx], dtype=np.float64)   # (M,)

        # predicted latent coefficients for displacement: first r entries of state
        alpha_pred = z_pred[k, 0:r].astype(np.float64)                   # (r,)

        # reconstruct predicted displacement via POD
        u_pred = mu.astype(np.float64) + U_r @ alpha_pred                # (M,)

        diff = u_pred - u_true
        e_L2 = np.linalg.norm(diff)
        n_true = np.linalg.norm(u_true) + 1e-12

        err_L2[k]     = e_L2
        err_rel_L2[k] = e_L2 / (n_true + 1e-6)
        if k == 0:
            err_rel_L2[k] = 0

    # keep a sample for quick inspection if needed (final time)
    result = {
        "t": t_roll,
        "err_L2": err_L2,
        "err_rel_L2": err_rel_L2,
        "u_pred_sample": u_pred,   # last step
        "u_true_sample": u_true,   # last step
        "r_modes": r_modes,
        "sim_id": sim_id,
    }

    print(f"[reconstruct] r={r_modes}, sim={sim_id}, T={T}")
    print(f"  Final disp L2 error:      {err_L2[-1]:.4e}")
    print(f"  Final disp relative L2:   {err_rel_L2[-1]:.4e}")

    return result


In [ ]:
def plot_displacement_reconstruction_errors(rec_dict):
    t         = rec_dict["t"]
    err_L2    = rec_dict["err_L2"]
    err_rel   = rec_dict["err_rel_L2"]
    r         = rec_dict["r_modes"]
    sim_id    = rec_dict["sim_id"]

    fig, ax = plt.subplots(1, 2, figsize=(10,4))

    ax[0].plot(t, err_L2)
    ax[0].set_xlabel("time [days]")
    ax[0].set_ylabel("||u_pred - u_true||_2")
    ax[0].set_title(f"L2 disp error (r={r}, sim={sim_id})")
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(t, err_rel)
    ax[1].set_xlabel("time [days]")
    ax[1].set_ylabel("relative L2 error")
    ax[1].set_title("Relative displacement error")
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_displacement_comparison_2d_simple(u_true,
                                           u_pred,
                                           node_xy,
                                           title_prefix="",
                                           xlim=(0, 300),
                                           ylim=(0, 300)):
    """
    Plot true vs predicted scalar displacement field (e.g. uz) on a 2D node grid,
    plus the error field (pred - true).

    Args
    ----
    u_true : (N,) array
        True scalar displacement values at each node.
    u_pred : (N,) array
        Predicted scalar displacement values at each node.
    node_xy : (N, 2) array
        Node coordinates [x, y] for each node, in the same ordering as u_true/u_pred.
    title_prefix : str, optional
        String to prepend to each subplot title (e.g. "uz field ").
    xlim : tuple, optional
        (xmin, xmax) limits for x-axis.
    ylim : tuple, optional
        (ymin, ymax) limits for y-axis.
    """
    u_true = np.asarray(u_true)
    u_pred = np.asarray(u_pred)
    node_xy = np.asarray(node_xy)

    assert u_true.shape == u_pred.shape, "u_true and u_pred must have same shape"
    assert node_xy.shape[0] == u_true.shape[0], "node_xy and u fields must have same length"

    err = u_pred - u_true
    x = node_xy[:, 0]
    y = node_xy[:, 1]

    # shared color scale for true/pred
    vmin = min(u_true.min(), u_pred.min())
    vmax = max(u_true.max(), u_pred.max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax_true, ax_pred, ax_err = axes

    # ---- TRUE ----
    sc0 = ax_true.scatter(x, y, c=u_true, s=5)
    ax_true.set_title(f"{title_prefix} true disp")
    ax_true.set_aspect("equal")
    ax_true.set_xlim(*xlim)
    ax_true.set_ylim(*ylim)
    ax_true.set_xlabel("x")
    ax_true.set_ylabel("y")
    plt.colorbar(sc0, ax=ax_true, fraction=0.046, pad=0.04)
    sc0.set_clim(vmin, vmax)

    # ---- PRED ----
    sc1 = ax_pred.scatter(x, y, c=u_pred, s=5)
    ax_pred.set_title(f"{title_prefix} pred disp")
    ax_pred.set_aspect("equal")
    ax_pred.set_xlim(*xlim)
    ax_pred.set_ylim(*ylim)
    ax_pred.set_xlabel("x")
    ax_pred.set_ylabel("y")
    plt.colorbar(sc1, ax=ax_pred, fraction=0.046, pad=0.04)
    sc1.set_clim(vmin, vmax)

    # ---- ERROR ----
    sc2 = ax_err.scatter(x, y, c=err, s=5)
    ax_err.set_title(f"{title_prefix} error (pred - true)")
    ax_err.set_aspect("equal")
    ax_err.set_xlim(*xlim)
    ax_err.set_ylim(*ylim)
    ax_err.set_xlabel("x")
    ax_err.set_ylabel("y")
    plt.colorbar(sc2, ax=ax_err, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


In [ ]:
def split_disp_block(u_flat):
    M = u_flat.shape[0]
    assert M % 3 == 0, "Displacement vector must be divisible by 3"
    N = M // 3

    ux = u_flat[0:N]
    uy = u_flat[N:2*N]
    uz = u_flat[2*N:3*N]
    return ux, uy, uz


In [ ]:
res = reconstruct_displacement_errors(r_modes=35, sim_id=aux["val_sims"][0])

u_true = res["u_true_sample"]
u_pred = res["u_pred_sample"]

ux_t, uy_t, uz_t = split_disp_block(u_true)
ux_p, uy_p, uz_p = split_disp_block(u_pred)

# sanity check: must match bottom layer count
assert uz_t.shape[0] == bottom_xy.shape[0], \
    f"{uz_t.shape[0]} displacement nodes but {bottom_xy.shape[0]} bottom-layer nodes!"


In [ ]:
plot_displacement_comparison_2d_simple(
    uz_t,
    uz_p,
    bottom_xy,
    title_prefix="uz field "
)


In [ ]:
def element_to_node_field(elem_field, elem_conn, n_nodes):
    """
    elem_field: (N_elem,) array of element scalar values
    elem_conn: (N_elem, n_nodes_per_elem) array of local node indices
    n_nodes: total number of nodes in the bottom mesh

    Returns:
        node_field: (n_nodes,) nodal-averaged field
    """
    elem_field = np.asarray(elem_field)
    node_field = np.zeros(n_nodes)
    counts     = np.zeros(n_nodes)

    for e, conn in enumerate(elem_conn):
        val = elem_field[e]
        for nid in conn:
            node_field[nid] += val
            counts[nid]     += 1

    # avoid divide by zero
    mask = counts > 0
    node_field[mask] /= counts[mask]

    return node_field


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_growth_field(node_xy, field_true, field_pred=None, title="λ_g field", cmap="viridis"):
    """
    node_xy: (N_nodes,2)
    field_true: (N_nodes,) nodal field
    field_pred: (N_nodes,) optional
    """
    node_xy = np.asarray(node_xy)
    field_true = np.asarray(field_true)

    x = node_xy[:, 0]
    y = node_xy[:, 1]

    if field_pred is None:
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        sc = ax.scatter(x, y, c=field_true, s=8, cmap=cmap)
        ax.set_aspect("equal")
        ax.set_title(title)
        plt.colorbar(sc, ax=ax)
        plt.show()
        return

    field_pred = np.asarray(field_pred)
    err = field_pred - field_true

    vmin = min(field_true.min(), field_pred.min())
    vmax = max(field_true.max(), field_pred.max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

    sc0 = axes[0].scatter(x, y, c=field_true, s=8, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title(title + " (true)")
    axes[0].set_aspect("equal")
    fig.colorbar(sc0, ax=axes[0])

    sc1 = axes[1].scatter(x, y, c=field_pred, s=8, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title(title + " (pred)")
    axes[1].set_aspect("equal")
    fig.colorbar(sc1, ax=axes[1])

    sc2 = axes[2].scatter(x, y, c=err, s=8, cmap="coolwarm")
    axes[2].set_title("error (pred - true)")
    axes[2].set_aspect("equal")
    fig.colorbar(sc2, ax=axes[2])

    plt.tight_layout()
    plt.show()


In [ ]:
# Time index to visualize (final time)
t_idx = -1   # or T-1

T = lamdag_hist_pred.shape[0]
assert lamdag_hist_true.shape[0] == T, "Predicted and true histories must have same T."

# predicted element-level fields at final time
final_elem_lamdag = lamdag_hist_pred[t_idx]      # (N_elem, 2)
lam_gx_elem_pred = final_elem_lamdag[:, 0]
lam_gy_elem_pred = final_elem_lamdag[:, 1]

# map predicted element fields → nodal fields
N_nodes_bottom = node_coords_bottom.shape[0]
lam_gx_node_pred = element_to_node_field(lam_gx_elem_pred, bottom_face_conn_local, N_nodes_bottom)
lam_gy_node_pred = element_to_node_field(lam_gy_elem_pred, bottom_face_conn_local, N_nodes_bottom)

# true nodal λ_g fields at the same time
lam_gx_node_true = lamdag_hist_true[t_idx, :, 0]   # (N_bottom,)
lam_gy_node_true = lamdag_hist_true[t_idx, :, 1]   # (N_bottom,)

# node xy

node_xy_bottom = node_coords_bottom[:, :2]

In [ ]:
print(T)
print(lamdag_hist_true.shape[0])

In [ ]:
# λ_gx comparison
plot_growth_field(
    node_xy=node_xy_bottom,
    field_true=lam_gx_node_true,
    field_pred=lam_gx_node_pred,
    title="λ_gx at final time"
)

# λ_gy comparison
plot_growth_field(
    node_xy=node_xy_bottom,
    field_true=lam_gy_node_true,
    field_pred=lam_gy_node_pred,
    title="λ_gy at final time"
)


In [ ]:
print(aux["val_sims"])

In [ ]:
import numpy as np
import zarr
import matplotlib.pyplot as plt

def eval_final_disp_for_val_sims(
    val_sims, r_modes, tol_rel=1e-3, eps_max=1e-12
):
    """
    For each sim_id in val_sims:
      - rollout NODE
      - reconstruct final predicted uz
      - load final true uz
      - compute displacement errors (with masking)
      - compute volume errors over the entire trajectory

    Returns
    -------
    all_metrics : list of dict
        Each dict has keys:
          "sim_id",
          # displacement metrics
          "L2_full", "rel_full",
          "L2_mask", "rel_mask",
          "mean_abs_mask", "max_abs_mask",
          "n_mask",
          # volume metrics
          "vol_L2",
          "vol_mean_err", "vol_mean_abs_err",
          "vol_rel_final", "vol_rel_mean"
    """

    # --- Load POD basis & mean, and FE snapshots ---
    g_disp = zarr.open_group(ZARR_DISP, mode="r")

    U_full = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_U_NAME}"], dtype=np.float32)
    mu     = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_MEAN_NAME}"], dtype=np.float32)
    U_r    = U_full[:, :r_modes]

    disp_snaps = g_disp[DISP_SNAP_DATASET]

    # true expander volume for all snapshots
    # (adapt key if your dataset name is different)
    exp_vol_all = np.asarray(g_disp["expander_volume"]["cvol"], dtype=np.float64)

    # mapping arrays
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    all_metrics = []

    for sim_id in val_sims:
        print(f"\n=== Validation sim {sim_id} ===")

        # 1) rollout
        roll = rollout_single_sim(r_modes=r_modes, sim_id=sim_id)
        z_pred = roll["z_pred"]
        T_roll = z_pred.shape[0]

        # predicted volume trajectory from the rollout
        # (adapt key if needed, e.g. "volume_pred")
        vol_pred_all = np.asarray(roll["vol_pred"], dtype=np.float64).reshape(-1)

        # 2) match sim → FE snapshot indices
        idx = np.where(sim_index == sim_id)[0]
        t_s = time_vals[idx]
        order = np.argsort(t_s)
        idx_sorted = idx[order]

        # effective length (in case of any mismatch)
        T_eff = min(T_roll, len(idx_sorted), len(vol_pred_all))
        if T_eff < T_roll or T_eff < len(vol_pred_all):
            print(f"  [WARN] Truncating volume/displacement from {T_roll} to {T_eff} steps.")
        idx_sorted_eff = idx_sorted[:T_eff]

        # ----- displacement metrics at final time -----

        snap_idx_final = idx_sorted_eff[T_eff - 1]

        # 3) true + predicted displacement vector (flat, 3N)
        u_true_flat = np.asarray(disp_snaps[:, snap_idx_final], dtype=np.float64)
        alpha_pred  = z_pred[T_eff - 1, :r_modes].astype(np.float64)
        u_pred_flat = mu + U_r @ alpha_pred

        # 4) split into blocks (per your trusted pipeline)
        ux_t, uy_t, uz_t = split_disp_block(u_true_flat)
        ux_p, uy_p, uz_p = split_disp_block(u_pred_flat)

        assert uz_t.shape[0] == bottom_xy.shape[0]

        # 5) --- Mask where true displacement is *meaningfully* non-zero ---
        uz_abs = np.abs(uz_t)
        max_uz = np.max(uz_abs)

        if max_uz < eps_max:
            # field basically zero → no masking
            print("  [WARN] Entire field ~zero. Using unmasked metrics.")
            mask = np.ones_like(uz_t, dtype=bool)
        else:
            # keep nodes that are NOT numerically near-zero
            thresh = tol_rel * max_uz
            mask = uz_abs >= thresh

        n_mask = np.count_nonzero(mask)
        if n_mask == 0:
            print("  [WARN] Mask removed all nodes, falling back to full field.")
            mask = np.ones_like(uz_t, dtype=bool)
            n_mask = mask.size

        uz_t_m = uz_t[mask]
        uz_p_m = uz_p[mask]
        err_m  = uz_p_m - uz_t_m

        # 6) Displacement metrics (final time)
        L2_full = np.linalg.norm(uz_p - uz_t)
        L2_mask = np.linalg.norm(err_m)

        rel_full = L2_full / (np.linalg.norm(uz_t) + 1e-12)
        rel_mask = L2_mask / (np.linalg.norm(uz_t_m) + 1e-12)

        max_abs_mask  = np.max(np.abs(err_m))
        mean_abs_mask = np.mean(np.abs(err_m))

        print(f"  Masked nodes: {n_mask}/{uz_t.size}")
        print(f"  Final uz L2 (full)      = {L2_full:.4e}")
        print(f"  Final uz relative (full)= {rel_full:.4e}")
        print(f"  Final uz L2 (masked)    = {L2_mask:.4e}")
        print(f"  Final uz relative(mask) = {rel_mask:.4e}")
        print(f"  Mean abs error (masked) = {mean_abs_mask:.4e}")
        print(f"  Max abs error  (masked) = {max_abs_mask:.4e}")

        # ----- volume metrics over the whole trajectory -----

        vol_true = exp_vol_all[idx_sorted_eff]          # (T_eff,)
        vol_pred = vol_pred_all[:T_eff]                 # (T_eff,)
        vol_err  = vol_pred - vol_true                  # (T_eff,)

        vol_L2           = np.linalg.norm(vol_err)
        vol_mean_err     = float(np.mean(vol_err))      # signed
        vol_mean_abs_err = float(np.mean(np.abs(vol_err)))

        vol_rel_final = float(
            np.abs(vol_err[-1]) / (np.abs(vol_true[-1]) + 1e-12)
        )
        vol_rel_mean = float(
            np.mean(np.abs(vol_err) / (np.abs(vol_true) + 1e-12))
        )

        print(f"  Volume L2 error (traj)  = {vol_L2:.4e}")
        print(f"  Volume mean error       = {vol_mean_err:.4e}")
        print(f"  Volume mean |error|     = {vol_mean_abs_err:.4e}")
        print(f"  Volume rel error final  = {vol_rel_final:.4e}")
        print(f"  Volume mean rel error   = {vol_rel_mean:.4e}")

        all_metrics.append({
            "sim_id": sim_id,
            # displacement stats
            "L2_full": L2_full,
            "rel_full": rel_full,
            "L2_mask": L2_mask,
            "rel_mask": rel_mask,
            "mean_abs_mask": mean_abs_mask,
            "max_abs_mask": max_abs_mask,
            "n_mask": int(n_mask),
            # volume stats
            "vol_L2": vol_L2,
            "vol_mean_err": vol_mean_err,
            "vol_mean_abs_err": vol_mean_abs_err,
            "vol_rel_final": vol_rel_final,
            "vol_rel_mean": vol_rel_mean,
        })

    return all_metrics


In [ ]:
def plot_final_disp_metrics_histograms(all_metrics, r_modes=None, bins=20):
    """
    Given all_metrics (list of dicts from eval_final_disp_for_val_sims),
    make histograms of the key final metrics across all sims.
    """

    import numpy as np
    import matplotlib.pyplot as plt

    # Collect each metric as a numpy array
    sim_ids      = np.array([m["sim_id"]         for m in all_metrics])
    L2_full      = np.array([m["L2_full"]        for m in all_metrics])
    rel_full     = np.array([m["rel_full"]       for m in all_metrics])
    L2_mask      = np.array([m["L2_mask"]        for m in all_metrics])
    rel_mask     = np.array([m["rel_mask"]       for m in all_metrics])
    mean_abs     = np.array([m["mean_abs_mask"]  for m in all_metrics])
    max_abs      = np.array([m["max_abs_mask"]   for m in all_metrics])
    n_mask       = np.array([m["n_mask"]         for m in all_metrics])

    # Optional: print summary stats
    def summarize(name, arr):
        print(f"{name}: mean={arr.mean():.4e}, median={np.median(arr):.4e}, "
              f"min={arr.min():.4e}, max={arr.max():.4e}")

    print("\n=== Summary over validation sims ===")
    print(f"Sims: {sim_ids}")
    summarize("L2_full",  L2_full)
    summarize("rel_full", rel_full)
    summarize("L2_mask",  L2_mask)
    summarize("rel_mask", rel_mask)
    summarize("mean_abs_mask", mean_abs)
    summarize("max_abs_mask",  max_abs)

    # Helper to plot one histogram
    def _plot_hist(values, title, xlabel, logx=False):
        plt.figure(figsize=(6, 4))
        if logx:
            # Avoid log of non-positive by clipping
            values = np.clip(values, a_min=1e-16, a_max=None)
            plt.hist(np.log10(values), bins=bins, alpha=0.8)
            plt.xlabel(f"log10({xlabel})")
        else:
            plt.hist(values, bins=bins, alpha=0.8)
            plt.xlabel(xlabel)

        if r_modes is not None:
            title = f"{title} (r={r_modes}, N={len(values)})"
        else:
            title = f"{title} (N={len(values)})"

        plt.title(title)
        plt.ylabel("Count")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Make histograms (tweak logx=True if distributions are very skewed)
    _plot_hist(rel_mask, "Final relative error (masked)", "rel_mask", logx=False)
    _plot_hist(rel_full, "Final relative error (full field)", "rel_full", logx=False)
    _plot_hist(L2_mask,  "Final L2 error (masked)", "L2_mask", logx=True)
    _plot_hist(L2_full,  "Final L2 error (full field)", "L2_full", logx=True)
    _plot_hist(mean_abs, "Final mean abs error (masked)", "mean_abs_mask", logx=True)
    _plot_hist(max_abs,  "Final max abs error (masked)", "max_abs_mask", logx=True)

    # If you want to see how many nodes are contributing:
    _plot_hist(n_mask,   "Number of masked nodes", "n_mask", logx=False)


In [ ]:
# Example: load from a checkpoint if you saved it
ckpt = torch.load("models/latent_velocity_model_r9.pt", map_location="cpu", weights_only=False)
val_sims = aux["val_sims"]  # fallback if not stored

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

def plot_final_disp_metrics_histograms(all_metrics, r_modes=None, bins=20,
                                       save_stats_path=None):
    """
    Given all_metrics (list of dicts from eval_final_disp_for_val_sims),
    make histograms of the key final metrics across all sims and compute
    summary statistics.

    Parameters
    ----------
    all_metrics : list[dict]
        Each dict should contain displacement metrics and, optionally,
        volume metrics (vol_L2, vol_mean_err, vol_mean_abs_err,
        vol_rel_final, vol_rel_mean).
    r_modes     : int or None
        POD dimension (for labeling only).
    bins        : int
        Number of histogram bins.
    save_stats_path : str or None
        If not None, save the summary statistics as JSON to this path.

    Returns
    -------
    stats : dict
        Nested dictionary of summary statistics for each metric.
    """

    # Collect each displacement metric as a numpy array
    sim_ids      = np.array([m["sim_id"]         for m in all_metrics])
    L2_full      = np.array([m["L2_full"]        for m in all_metrics])
    rel_full     = np.array([m["rel_full"]       for m in all_metrics])
    L2_mask      = np.array([m["L2_mask"]        for m in all_metrics])
    rel_mask     = np.array([m["rel_mask"]       for m in all_metrics])
    mean_abs     = np.array([m["mean_abs_mask"]  for m in all_metrics])
    max_abs      = np.array([m["max_abs_mask"]   for m in all_metrics])
    n_mask       = np.array([m["n_mask"]         for m in all_metrics])

    # Volume metrics (if present)
    has_volume = "vol_L2" in all_metrics[0]
    if has_volume:
        vol_L2           = np.array([m["vol_L2"]           for m in all_metrics])
        vol_mean_err     = np.array([m["vol_mean_err"]     for m in all_metrics])
        vol_mean_abs_err = np.array([m["vol_mean_abs_err"] for m in all_metrics])
        vol_rel_final    = np.array([m["vol_rel_final"]    for m in all_metrics])
        vol_rel_mean     = np.array([m["vol_rel_mean"]     for m in all_metrics])

    # ---- summary statistics helper ----
    def compute_stats(arr):
        return {
            "mean":   float(arr.mean()),
            "median": float(np.median(arr)),
            "std":    float(arr.std()),
            "min":    float(arr.min()),
            "max":    float(arr.max()),
            "p90":    float(np.percentile(arr, 90)),
            "p95":    float(np.percentile(arr, 95)),
        }

    metrics_dict = {
        "L2_full":       compute_stats(L2_full),
        "rel_full":      compute_stats(rel_full),
        "L2_mask":       compute_stats(L2_mask),
        "rel_mask":      compute_stats(rel_mask),
        "mean_abs_mask": compute_stats(mean_abs),
        "max_abs_mask":  compute_stats(max_abs),
        "n_mask":        compute_stats(n_mask),
    }

    if has_volume:
        metrics_dict.update({
            "vol_L2":            compute_stats(vol_L2),
            "vol_mean_err":      compute_stats(vol_mean_err),
            "vol_mean_abs_err":  compute_stats(vol_mean_abs_err),
            "vol_rel_final":     compute_stats(vol_rel_final),
            "vol_rel_mean":      compute_stats(vol_rel_mean),
        })

    stats = {
        "r_modes": int(r_modes) if r_modes is not None else None,
        "N": int(len(rel_mask)),
        "sim_ids": sim_ids.tolist(),
        "metrics": metrics_dict,
    }

    # ---- print a concise summary, with emphasis on rel_mask & volume rel ----
    print("\n=== Summary over validation sims ===")
    if r_modes is not None:
        print(f"r = {r_modes}, N_sims = {len(sim_ids)}")
    else:
        print(f"N_sims = {len(sim_ids)}")
    print(f"Sims: {sim_ids}")

    def print_stats(name, s):
        print(
            f"{name:15s}: "
            f"mean={s['mean']:.4e}, median={s['median']:.4e}, "
            f"std={s['std']:.4e}, min={s['min']:.4e}, max={s['max']:.4e}, "
            f"p90={s['p90']:.4e}, p95={s['p95']:.4e}"
        )

    # displacement
    print_stats("rel_mask", stats["metrics"]["rel_mask"])
    print_stats("rel_full", stats["metrics"]["rel_full"])
    print_stats("L2_mask",  stats["metrics"]["L2_mask"])
    print_stats("L2_full",  stats["metrics"]["L2_full"])
    print_stats("mean_abs", stats["metrics"]["mean_abs_mask"])
    print_stats("max_abs",  stats["metrics"]["max_abs_mask"])

    # volume (if available)
    if has_volume:
        print_stats("vol_rel_final", stats["metrics"]["vol_rel_final"])
        print_stats("vol_rel_mean",  stats["metrics"]["vol_rel_mean"])
        print_stats("vol_L2",        stats["metrics"]["vol_L2"])
        print_stats("vol_mean_err",  stats["metrics"]["vol_mean_err"])
        print_stats("vol_mean_abs",  stats["metrics"]["vol_mean_abs_err"])

    # ---- histogram helper ----
    def _plot_hist(values, title, xlabel, logx=False):
        plt.figure(figsize=(6, 4))
        if logx:
            # Avoid log of non-positive by clipping
            values = np.clip(values, a_min=1e-16, a_max=None)
            plt.hist(np.log10(values), bins=bins, alpha=0.8)
            plt.xlabel(f"log10({xlabel})")
        else:
            plt.hist(values, bins=bins, alpha=0.8)
            plt.xlabel(xlabel)

        if r_modes is not None:
            title = f"{title} (r={r_modes}, N={len(values)})"
        else:
            title = f"{title} (N={len(values)})"

        plt.title(title)
        plt.ylabel("Count")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ---- Displacement histograms ----
    _plot_hist(rel_mask, "Final relative error (masked)", "rel_mask", logx=False)
    _plot_hist(rel_full, "Final relative error (full field)", "rel_full", logx=False)
    _plot_hist(L2_mask,  "Final L2 error (masked)", "L2_mask", logx=True)
    _plot_hist(L2_full,  "Final L2 error (full field)", "L2_full", logx=True)
    _plot_hist(mean_abs, "Final mean abs error (masked)", "mean_abs_mask", logx=True)
    _plot_hist(max_abs,  "Final max abs error (masked)", "max_abs_mask", logx=True)
    _plot_hist(n_mask,   "Number of masked nodes", "n_mask", logx=False)

    # ---- Volume histograms (if available) ----
    if has_volume:
        _plot_hist(vol_rel_final, "Final relative volume error", "vol_rel_final", logx=False)
        _plot_hist(vol_rel_mean,  "Mean relative volume error",  "vol_rel_mean",  logx=False)
        _plot_hist(vol_L2,        "Volume L2 error (trajectory)", "vol_L2",       logx=True)
        _plot_hist(vol_mean_abs_err, "Mean |volume error|", "vol_mean_abs_err",   logx=True)
        _plot_hist(vol_mean_err,     "Mean signed volume error", "vol_mean_err",  logx=True)

    # ---- optional: save to disk ----
    if save_stats_path is not None:
        with open(save_stats_path, "w") as f:
            json.dump(stats, f, indent=2)
        print(f"\nSaved stats to {save_stats_path}")

    return stats


In [ ]:
val_sims = aux["val_sims"]  # whatever your validation sims are
r_modes = 9

all_metrics = eval_final_disp_for_val_sims(val_sims, r_modes)


In [ ]:
stats = plot_final_disp_metrics_histograms(
    all_metrics,
    r_modes=r_modes,
    bins=20,
    save_stats_path=f"disp_error_stats_r{r}.json"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import zarr

def visualize_mask_for_sim(
    sim_id,
    r_modes,
    tol_rel=1e-3,  # same as in your eval function if you're using relative mask
    eps_max=1e-12
):
    """
    For a single sim_id:
      - rollout NODE
      - reconstruct final uz
      - apply the same mask logic as in eval_final_disp_for_val_sims
      - plot which nodes are kept vs masked-out on bottom_xy
    """

    # --- Load POD basis & mean, and FE snapshots ---
    g_disp = zarr.open_group(ZARR_DISP, mode="r")

    U_full = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_U_NAME}"], dtype=np.float32)
    mu     = np.asarray(g_disp[f"{DISP_POD_GROUP}/{DISP_POD_MEAN_NAME}"], dtype=np.float32)
    U_r    = U_full[:, :r_modes]

    disp_snaps = g_disp[DISP_SNAP_DATASET]

    # mapping arrays
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    print(f"\n=== Visualizing mask for sim {sim_id} ===")

    # 1) rollout
    roll = rollout_single_sim(r_modes=r_modes, sim_id=sim_id)
    z_pred = roll["z_pred"]
    T_roll = z_pred.shape[0]

    # 2) match sim → FE snapshot indices
    idx = np.where(sim_index == sim_id)[0]
    t_s = time_vals[idx]
    order = np.argsort(t_s)
    idx_sorted = idx[order]

    snap_idx_final = idx_sorted[T_roll - 1]
    t_final = t_s[order][T_roll - 1]

    # 3) true + predicted displacement vector (flat, 3N)
    u_true_flat = np.asarray(disp_snaps[:, snap_idx_final], dtype=np.float64)
    alpha_pred  = z_pred[-1, :r_modes].astype(np.float64)
    u_pred_flat = mu + U_r @ alpha_pred

    # 4) split into blocks (per your trusted pipeline)
    ux_t, uy_t, uz_t = split_disp_block(u_true_flat)
    ux_p, uy_p, uz_p = split_disp_block(u_pred_flat)

    assert uz_t.shape[0] == bottom_xy.shape[0]

    # 5) --- Mask where true displacement is *meaningfully* non-zero ---
    uz_abs = np.abs(uz_t)
    max_uz = np.max(uz_abs)

    if max_uz < eps_max:
        print("  [WARN] Entire field ~zero. Using unmasked metrics.")
        mask = np.ones_like(uz_t, dtype=bool)
    else:
        thresh = tol_rel * max_uz
        mask = uz_abs >= thresh

    n_mask = np.count_nonzero(mask)
    print(f"  Max |uz| = {max_uz:.4e}")
    print(f"  Threshold = {tol_rel} * max = {thresh:.4e}")
    print(f"  Masked-in nodes:  {n_mask}")
    print(f"  Masked-out nodes: {uz_t.size - n_mask} / {uz_t.size}")

    # 6) Make a plot showing where mask is applied
    x = bottom_xy[:, 0]
    y = bottom_xy[:, 1]

    plt.figure(figsize=(6, 6))
    plt.scatter(x[~mask], y[~mask],
                s=4, alpha=0.3, label="masked-out (near-zero uz)")
    plt.scatter(x[mask], y[mask],
                s=4, alpha=0.8, label="kept (non-zero uz)")

    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("x (bottom layer)")
    plt.ylabel("y (bottom layer)")
    plt.title(f"Mask on bottom layer nodes\nsim {sim_id}, t ≈ {t_final:.3f}")
    plt.legend(loc="best")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 7) Optional: plot uz magnitude with masked nodes faded
    plt.figure(figsize=(6, 6))
    sc = plt.scatter(x, y, c=uz_abs, s=4, cmap="viridis")
    plt.scatter(x[~mask], y[~mask],
                s=8, facecolors="none", edgecolors="red", linewidths=0.7,
                label="masked-out")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("x (bottom layer)")
    plt.ylabel("y (bottom layer)")
    plt.title(f"|uz_true| with masked-out nodes highlighted\nsim {sim_id}, t ≈ {t_final:.3f}")
    plt.colorbar(sc, label="|uz_true|")
    plt.legend(loc="best")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
r_modes = 20
sim_to_debug = val_sims[0]  # or pick any specific sim_id

visualize_mask_for_sim(sim_to_debug, r_modes=r_modes, tol_rel=1e-3)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

def load_stats(path):
    with open(path, "r") as f:
        return json.load(f)

# Which r to compare
r_list = [4, 5, 6, 7, 8, 9, 10]

# Map r -> stats dict
stats_by_r = {}
for r in r_list:
    fname = f"disp_error_stats_r{r}.json"  # change if your filenames differ
    stats_by_r[r] = load_stats(fname)
    print(f"Loaded {fname}")


In [ ]:
rows = []
for r in r_list:
    m = stats_by_r[r]["metrics"]
    rel_mask = m["rel_mask"]
    rel_full = m["rel_full"]
    L2_mask  = m["L2_mask"]
    L2_full  = m["L2_full"]

    rows.append({
        "r": r,
        "rel_mask_mean":   rel_mask["mean"],
        "rel_mask_median": rel_mask["median"],
        "rel_mask_p90":    rel_mask["p90"],
        "rel_mask_p95":    rel_mask["p95"],
        "rel_full_mean":   rel_full["mean"],
        "rel_full_median": rel_full["median"],
        "L2_mask_mean":    L2_mask["mean"],
        "L2_mask_p95":     L2_mask["p95"],
        "L2_full_mean":    L2_full["mean"],
        "L2_full_p95":     L2_full["p95"],
    })

disp_df = pd.DataFrame(rows).set_index("r").sort_index()
print(disp_df.round(4))


In [ ]:
plt.figure(figsize=(6,4))
plt.bar(disp_df.index - 0.3/2, disp_df["rel_mask_median"], width=0.3, label="median")
plt.bar(disp_df.index + 0.3/2, disp_df["rel_mask_p95"],  width=0.3, label="p95")

plt.xlabel("r (POD modes)")
plt.ylabel("masked relative error")
plt.title("Final uz masked relative error vs r")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
vol_rows = []
for r in r_list:
    m = stats_by_r[r]["metrics"]
    vol_rel_mean  = m.get("vol_rel_mean",  None)
    vol_rel_final = m.get("vol_rel_final", None)

    if vol_rel_mean is None:
        continue

    vol_rows.append({
        "r": r,
        "vol_rel_mean_mean":   vol_rel_mean["mean"],
        "vol_rel_mean_p95":    vol_rel_mean["p95"],
        "vol_rel_final_mean":  vol_rel_final["mean"],
        "vol_rel_final_p95":   vol_rel_final["p95"],
    })

vol_df = pd.DataFrame(vol_rows).set_index("r").sort_index()
print(vol_df.round(4))


In [ ]:
import numpy as np
import zarr

def eval_growth_metrics_for_val_sims(
    val_sims,
    r_modes,
    tol_rel=1e-3,
    eps_max=1e-12,
    ip_zarr_path="ip_nodal.zarr",
    gx_key="snaps_SDV1_bottom",
    gy_key="snaps_SDV4_bottom",
):
    """
    Evaluate growth prediction error for each sim in val_sims, using the
    SAME spatial mask as the displacement evaluation (built from final-time uz).

    Mask definition (per sim):
      - Take true uz at final time on the bottom layer.
      - Build boolean mask:
            mask = |uz_true| >= tol_rel * max(|uz_true|)
      - Any nodes masked out for uz are ALSO masked out for growth.

    Growth:
      - True growth (bottom nodal) is read from ip_nodal.zarr:
            lamgx_all = g_ip[gx_key][:]  # shape (N_bottom, S_snapshots)
            lamgy_all = g_ip[gy_key][:]  # shape (N_bottom, S_snapshots)
      - Predicted growth is per element in the rollout:
            lamdag_hist_pred = roll["lamdag_hist_pred"]  # shape (T_roll, N_elem, 2)
        We map this to nodal values on the bottom by averaging element values
        over incident elements for each node.

    Returns
    -------
    all_growth_metrics : list[dict]
        For each sim_id, a dict with:

            "sim_id"
            "n_mask"

            # aggregated over all times & masked nodes
            "g_L2_all"
            "g_rel_all"
            "g_mean_abs_all"
            "g_rel_mean_all"

            # final time slice (still using the same mask)
            "g_L2_final"
            "g_rel_final"
    """

    # --- Open true growth snapshots (bottom layer, nodal) ---
    g_ip = zarr.open(ip_zarr_path, mode="r")
    lamgx_all = np.asarray(g_ip[gx_key][:], dtype=np.float64)  # (N_bottom, S_snapshots)
    lamgy_all = np.asarray(g_ip[gy_key][:], dtype=np.float64)  # (N_bottom, S_snapshots)

    # --- Open displacement snapshots (for mask) ---
    g_disp = zarr.open_group(ZARR_DISP, mode="r")
    disp_snaps = np.asarray(g_disp[DISP_SNAP_DATASET], dtype=np.float64)  # (3*N_bottom, S)

    # mapping arrays (same as everywhere else)
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    # Global geometry info (assumed already defined in your module)
    # bottom_xy: (N_bottom, 2) or (N_bottom, 3) with bottom layer nodes
    # bottom_face_conn_local: (N_elem, nodes_per_elem) with local node indices [0..N_bottom-1]
    N_bottom = bottom_xy.shape[0]
    N_elem   = bottom_face_conn_local.shape[0]

    all_growth_metrics = []

    for sim_id in val_sims:
        print(f"\n=== Growth metrics (disp-based mask) for sim {sim_id} ===")

        # 1) rollout (must include per-element growth history)
        roll = rollout_single_sim(r_modes=r_modes, sim_id=sim_id)

        if "lamdag_hist_pred" not in roll:
            raise KeyError("roll['lamdag_hist_pred'] not found. "
                           "Make sure you saved lamdag_hist_pred into the rollout dict.")

        lamdag_hist_pred = np.asarray(roll["lamdag_hist_pred"], dtype=np.float64)
        # lamdag_hist_pred shape: (T_roll, N_elem, 2) → [:, :, 0] = λ_gx, [:, :, 1] = λ_gy
        T_roll = lamdag_hist_pred.shape[0]
        assert lamdag_hist_pred.shape[1] == N_elem, "Mismatch: N_elem vs lamdag_hist_pred.shape[1]"

        # 2) match sim → snapshot indices and sort by time
        idx = np.where(sim_index == sim_id)[0]
        t_s = time_vals[idx]
        order = np.argsort(t_s)
        idx_sorted = idx[order]

        # effective time length (align with rollout & snapshots)
        T_eff = min(T_roll, len(idx_sorted))
        if T_eff < T_roll or T_eff < len(idx_sorted):
            print(f"  [WARN] Truncating trajectories from {T_roll} to {T_eff} steps.")
        idx_sorted_eff = idx_sorted[:T_eff]

        # ----------------------------------------------------
        # 3) Build the displacement-based mask (from final uz)
        # ----------------------------------------------------
        snap_idx_final = idx_sorted_eff[T_eff - 1]

        # true displacement at final time, flat 3N vector
        u_true_flat = disp_snaps[:, snap_idx_final]

        # Your trusted splitter: returns bottom layer uz in same order as bottom nodes
        ux_t, uy_t, uz_t = split_disp_block(u_true_flat)  # uz_t shape (N_bottom,)
        assert uz_t.shape[0] == N_bottom, "bottom layer size mismatch"

        uz_abs = np.abs(uz_t)
        max_uz = np.max(uz_abs)

        if max_uz < eps_max:
            print("  [WARN] Final uz field ~zero; using unmasked metrics.")
            mask = np.ones_like(uz_t, dtype=bool)
        else:
            thresh = tol_rel * max_uz
            mask = uz_abs >= thresh

        if not np.any(mask):
            print("  [WARN] Displacement mask removed all nodes; using full field.")
            mask = np.ones_like(uz_t, dtype=bool)

        n_mask = int(np.count_nonzero(mask))
        print(f"  Displacement-based mask keeps {n_mask}/{uz_t.size} bottom nodes.")

        # ----------------------------------------------------
        # 4) True growth trajectories for this sim (nodal, bottom)
        # ----------------------------------------------------
        # lamgx_all / lamgy_all: (N_bottom, S_snapshots)
        # select columns idx_sorted_eff -> (N_bottom, T_eff), then transpose
        lamgx_true = lamgx_all[:, idx_sorted_eff].T  # (T_eff, N_bottom)
        lamgy_true = lamgy_all[:, idx_sorted_eff].T  # (T_eff, N_bottom)

        # ----------------------------------------------------
        # 5) Map per-element predicted growth to nodal values
        # ----------------------------------------------------
        # lamdag_hist_pred: (T_roll, N_elem, 2) → restrict to T_eff
        lamdag_eff = lamdag_hist_pred[:T_eff, :, :]  # (T_eff, N_elem, 2)

        # Allocate nodal arrays
        lamgx_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        lamgy_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        counts = np.zeros(N_bottom, dtype=np.int32)

        for k in range(T_eff):
            lamgx_pred_nodal[k, :] = 0.0
            lamgy_pred_nodal[k, :] = 0.0
            counts[:] = 0

            # Accumulate element growth onto nodes
            lam_k = lamdag_eff[k]  # (N_elem, 2)
            for e in range(N_elem):
                lamgx_e = lam_k[e, 0]
                lamgy_e = lam_k[e, 1]
                nodes_e = bottom_face_conn_local[e]  # indices into [0..N_bottom-1]
                for nid in nodes_e:
                    lamgx_pred_nodal[k, nid] += lamgx_e
                    lamgy_pred_nodal[k, nid] += lamgy_e
                    counts[nid] += 1

            # Average over contributing elements
            nonzero = counts > 0
            lamgx_pred_nodal[k, nonzero] /= counts[nonzero]
            lamgy_pred_nodal[k, nonzero] /= counts[nonzero]

            # For nodes with no contributing elements (if any), set λ = 1 (no growth)
            zero = ~nonzero
            if np.any(zero):
                lamgx_pred_nodal[k, zero] = 1.0
                lamgy_pred_nodal[k, zero] = 1.0

        # ----------------------------------------------------
        # 6) Work in λ_g - 1 space and apply SAME mask
        # ----------------------------------------------------
        gx_true_dev = lamgx_true - 1.0      # (T_eff, N_bottom)
        gy_true_dev = lamgy_true - 1.0
        gx_pred_dev = lamgx_pred_nodal - 1.0
        gy_pred_dev = lamgy_pred_nodal - 1.0

        # Apply mask along node axis
        gx_true_dev_m = gx_true_dev[:, mask]   # (T_eff, n_mask)
        gy_true_dev_m = gy_true_dev[:, mask]
        gx_pred_dev_m = gx_pred_dev[:, mask]
        gy_pred_dev_m = gy_pred_dev[:, mask]

        # Stack components and flatten over time & nodes
        g_true_dev_all = np.concatenate([gx_true_dev_m, gy_true_dev_m], axis=1)  # (T_eff, 2*n_mask)
        g_pred_dev_all = np.concatenate([gx_pred_dev_m, gy_pred_dev_m], axis=1)

        g_true_flat = g_true_dev_all.reshape(-1)
        g_pred_flat = g_pred_dev_all.reshape(-1)
        g_err_flat  = g_pred_flat - g_true_flat

        # ----------------------------------------------------
        # 7) Global growth metrics (all times & masked nodes)
        # ----------------------------------------------------
        g_L2_all       = float(np.linalg.norm(g_err_flat))
        g_rel_all      = float(g_L2_all / (np.linalg.norm(g_true_flat) + 1e-12))
        g_mean_abs_all = float(np.mean(np.abs(g_err_flat)))
        g_rel_mean_all = float(
            np.mean(np.abs(g_err_flat) / (np.abs(g_true_flat) + 1e-12))
        )

        # ----------------------------------------------------
        # 8) Final-time growth metrics (last time slice, same mask)
        # ----------------------------------------------------
        g_true_final = g_true_dev_all[-1, :]   # (2*n_mask,)
        g_pred_final = g_pred_dev_all[-1, :]
        g_err_final  = g_pred_final - g_true_final

        g_L2_final  = float(np.linalg.norm(g_err_final))
        g_rel_final = float(
            g_L2_final / (np.linalg.norm(g_true_final) + 1e-12)
        )

        print(f"  Growth L2 (all times, masked)   = {g_L2_all:.4e}")
        print(f"  Growth rel (all times, masked)  = {g_rel_all:.4e}")
        print(f"  Growth mean |err| (all, masked) = {g_mean_abs_all:.4e}")
        print(f"  Growth mean rel (all, masked)   = {g_rel_mean_all:.4e}")
        print(f"  Growth L2 (final, masked)       = {g_L2_final:.4e}")
        print(f"  Growth rel (final, masked)      = {g_rel_final:.4e}")

        all_growth_metrics.append({
            "sim_id": sim_id,
            "n_mask": n_mask,
            # aggregated over time & masked nodes
            "g_L2_all": g_L2_all,
            "g_rel_all": g_rel_all,
            "g_mean_abs_all": g_mean_abs_all,
            "g_rel_mean_all": g_rel_mean_all,
            # final-time metrics
            "g_L2_final": g_L2_final,
            "g_rel_final": g_rel_final,
        })

    return all_growth_metrics


In [ ]:
import numpy as np
import zarr
import time

def precompute_lamdag_for_val_sims(
    val_sims,
    r_modes,
    out_zarr_path="lamdag_pred_r9.zarr",
):
    """
    For each sim_id in val_sims:
      - rollout NODE
      - reconstruct nodal displacements
      - integrate growth over time using integrate_growth
      - save lamdag_hist_pred (T, N_elem, 2) into a Zarr store
      - store timing information for performance evaluation

    Stored timing metrics (in seconds):
        rollout_time
        reconstruction_time
        growth_integration_time
        total_sim_time
    """

    # Open/create output Zarr store (overwrite)
    root = zarr.open(out_zarr_path, mode="w")

    # Geometry info
    N_elem = bottom_face_conn_local.shape[0]

    for sim_id in val_sims:
        print(f"\n=== Precomputing lamdag_hist_pred for sim {sim_id} ===")

        sim_start_time = time.perf_counter()

        # 1) NODE rollout
        t0 = time.perf_counter()
        roll = rollout_single_sim(
            r_modes=r_modes,
            sim_id=sim_id,
            max_steps=None,
        )
        rollout_time = time.perf_counter() - t0

        # 2) Reconstruct nodal displacements over time
        t1 = time.perf_counter()
        t, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
            roll=roll,
            r_modes=r_modes,
        )
        reconstruction_time = time.perf_counter() - t1

        T = len(t)

        # 3) Get job-specific growth parameters
        job_id, theta_crit, k1, k2 = get_growth_params_for_sim(sim_id)
        print(f"  job {job_id}: theta_crit={theta_crit:.4f}, k1={k1:.4f}, k2={k2:.4f}")

        # 4) Initialize predicted growth field on bottom faces
        elem_lamdag = np.ones((N_elem, 2), dtype=float)
        lamdag_hist_pred = np.zeros((T, N_elem, 2), dtype=float)
        lamdag_hist_pred[0] = elem_lamdag.copy()

        # 5) Integrate growth over time
        t2 = time.perf_counter()
        for k in range(T - 1):
            dt_k = t[k+1] - t[k]
            node_u_k = u_nodes_seq[k]  # (N_nodes, 3)

            elem_lamdag = integrate_growth(
                node_coords=node_coords_bottom,
                elem_conn=bottom_face_conn_local,
                node_u=node_u_k,
                elem_lamdag=elem_lamdag,
                k1=k1,
                k2=k2,
                time=t[k],
                theta_crit=theta_crit,
                dt=dt_k,
            )
            lamdag_hist_pred[k+1] = elem_lamdag.copy()

        growth_integration_time = time.perf_counter() - t2

        total_sim_time = time.perf_counter() - sim_start_time

        # 6) Save to Zarr under sim_<sim_id>
        grp_name = f"sim_{sim_id}"
        if grp_name in root:
            del root[grp_name]

        g_sim = root.create_group(grp_name)

        # Save arrays
        g_sim.create_array("lamdag_hist_pred", data=lamdag_hist_pred)
        g_sim.create_array("t", data=t)

        # Save metadata
        g_sim.attrs["job_id"] = int(job_id)
        g_sim.attrs["theta_crit"] = float(theta_crit)
        g_sim.attrs["k1"] = float(k1)
        g_sim.attrs["k2"] = float(k2)

        # Save timing metrics
        g_sim.attrs["rollout_time_sec"] = float(rollout_time)
        g_sim.attrs["reconstruction_time_sec"] = float(reconstruction_time)
        g_sim.attrs["growth_integration_time_sec"] = float(growth_integration_time)
        g_sim.attrs["total_sim_time_sec"] = float(total_sim_time)

        print(f"  Saved lamdag_hist_pred for sim {sim_id} with shape {lamdag_hist_pred.shape}")
        print(f"  Timing (sec): rollout={rollout_time:.4f}, "
              f"recon={reconstruction_time:.4f}, "
              f"growth={growth_integration_time:.4f}, "
              f"total={total_sim_time:.4f}")

    print(f"\nDone. All lamdag histories saved to {out_zarr_path}")


In [ ]:
# r = 9
# precompute_lamdag_for_val_sims(val_sims, r_modes=r, out_zarr_path="lamdag_pred_r9.zarr")

In [ ]:
r = 9

growth_metrics_r9 = eval_growth_metrics_for_val_sims(
    val_sims[:120],
    r_modes=r,
    tol_rel=1e-3,
)

import numpy as np
g_rel_all_vals   = np.array([m["g_rel_all"]   for m in growth_metrics_r9])
g_rel_final_vals = np.array([m["g_rel_final"] for m in growth_metrics_r9])

print("Growth rel (all times, masked): mean =", g_rel_all_vals.mean(), "std =", g_rel_all_vals.std())
print("Growth rel (final, masked):     mean =", g_rel_final_vals.mean(), "std =", g_rel_final_vals.std())


In [ ]:
import numpy as np
import zarr
import matplotlib.pyplot as plt

def eval_growth_metrics_for_val_sims_from_zarr(
    val_sims,
    r_modes,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
    tol_rel=1e-3,
    eps_max=1e-12,
    ip_zarr_path="ip_nodal.zarr",
    gx_key="snaps_SDV1_bottom",
    gy_key="snaps_SDV4_bottom",
    plot_growth_fields=False,
    max_plot_sims=5,
):
    """
    Evaluate growth metrics for a set of validation simulations, using
    lamdag histories precomputed and stored in a Zarr store.

    - True growth fields are read from ip_zarr_path (lamgx_all, lamgy_all).
    - Predicted growth histories are read from lamdag_zarr_path.
    - A displacement-based mask (from final uz) is used for error metrics.
    - Optionally, plots true vs predicted λ_gx, λ_gy (unmasked) at final time.

    Metrics per sim:
      - g_L2_all, g_rel_all, g_mean_abs_all, g_rel_mean_all
      - g_L2_final, g_rel_final
      - area_gain_true, area_gain_pred, area_gain_err_abs, area_gain_err_rel
        (now in cm^2, using undeformed element area = 0.25 cm^2)
      - n_mask
    """

    # True nodal growth
    g_ip = zarr.open(ip_zarr_path, mode="r")
    lamgx_all = np.asarray(g_ip[gx_key][:], dtype=np.float64)  # (N_bottom, S)
    lamgy_all = np.asarray(g_ip[gy_key][:], dtype=np.float64)

    # Displacement snapshots (for mask)
    g_disp = zarr.open_group(ZARR_DISP, mode="r")
    disp_snaps = np.asarray(g_disp[DISP_SNAP_DATASET], dtype=np.float64)

    # Mapping arrays
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    # Geometry
    N_bottom = bottom_xy.shape[0]
    N_elem   = bottom_face_conn_local.shape[0]

    # Open lamdag store
    g_lam = zarr.open(lamdag_zarr_path, mode="r")

    all_growth_metrics = []
    n_plotted = 0  # how many sims we've plotted so far

    for sim_id in val_sims:
        print(f"\n=== Growth metrics (disp-based mask) for sim {sim_id} ===")

        grp_name = f"sim_{sim_id}"
        if grp_name not in g_lam:
            raise KeyError(
                f"{grp_name} not found in {lamdag_zarr_path}. "
                "Did you run precompute_lamdag_for_val_sims first?"
            )

        g_sim = g_lam[grp_name]
        lamdag_hist_pred = np.asarray(g_sim["lamdag_hist_pred"][:], dtype=np.float64)
        # lamdag_hist_pred shape: (T_roll, N_elem, 2)
        T_roll = lamdag_hist_pred.shape[0]

        # ---- Match sim → FE snapshots ----
        idx = np.where(sim_index == sim_id)[0]
        t_s = time_vals[idx]
        order = np.argsort(t_s)
        idx_sorted = idx[order]

        T_eff = min(T_roll, len(idx_sorted))
        if T_eff < T_roll or T_eff < len(idx_sorted):
            print(f"  [WARN] Truncating trajectories from {T_roll} to {T_eff} steps.")
        idx_sorted_eff = idx_sorted[:T_eff]
        lamdag_eff = lamdag_hist_pred[:T_eff, :, :]  # (T_eff, N_elem, 2)

        # ---- Build displacement-based mask from final uz ----
        snap_idx_final = idx_sorted_eff[T_eff - 1]
        u_true_flat = disp_snaps[:, snap_idx_final]
        ux_t, uy_t, uz_t = split_disp_block(u_true_flat)
        assert uz_t.shape[0] == N_bottom

        uz_abs = np.abs(uz_t)
        max_uz = np.max(uz_abs)
        if max_uz < eps_max:
            print("  [WARN] Final uz ~0; using full field for mask.")
            mask = np.ones_like(uz_t, dtype=bool)
        else:
            thresh = tol_rel * max_uz
            mask = uz_abs >= thresh
        if not np.any(mask):
            mask = np.ones_like(uz_t, dtype=bool)

        n_mask = int(np.count_nonzero(mask))
        print(f"  Mask keeps {n_mask}/{uz_t.size} bottom nodes.")

        # ---- True nodal growth for this sim ----
        lamgx_true = lamgx_all[:, idx_sorted_eff].T  # (T_eff, N_bottom)
        lamgy_true = lamgy_all[:, idx_sorted_eff].T

        # ---- Map element lamdag → nodal growth ----
        lamgx_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        lamgy_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        counts = np.zeros(N_bottom, dtype=np.int32)

        for k in range(T_eff):
            lamgx_pred_nodal[k, :] = 0.0
            lamgy_pred_nodal[k, :] = 0.0
            counts[:] = 0

            lam_k = lamdag_eff[k]  # (N_elem, 2)
            for e in range(N_elem):
                lamgx_e = lam_k[e, 0]
                lamgy_e = lam_k[e, 1]
                nodes_e = bottom_face_conn_local[e]
                for nid in nodes_e:
                    lamgx_pred_nodal[k, nid] += lamgx_e
                    lamgy_pred_nodal[k, nid] += lamgy_e
                    counts[nid] += 1

            nonzero = counts > 0
            lamgx_pred_nodal[k, nonzero] /= counts[nonzero]
            lamgy_pred_nodal[k, nonzero] /= counts[nonzero]

            zero = ~nonzero
            if np.any(zero):
                lamgx_pred_nodal[k, zero] = 1.0
                lamgy_pred_nodal[k, zero] = 1.0

        # ---- Use λ directly (no λ - 1) for metrics, with mask ----
        gx_true_m = lamgx_true[:, mask]        # (T_eff, n_masked)
        gy_true_m = lamgy_true[:, mask]
        gx_pred_m = lamgx_pred_nodal[:, mask]
        gy_pred_m = lamgy_pred_nodal[:, mask]

        g_true_all = np.concatenate([gx_true_m, gy_true_m], axis=1)
        g_pred_all = np.concatenate([gx_pred_m, gy_pred_m], axis=1)

        g_true_flat = g_true_all.reshape(-1)
        g_pred_flat = g_pred_all.reshape(-1)
        g_err_flat  = g_pred_flat - g_true_flat

        g_L2_all       = float(np.linalg.norm(g_err_flat))
        g_rel_all      = float(g_L2_all / (np.linalg.norm(g_true_flat) + 1e-12))
        g_mean_abs_all = float(np.mean(np.abs(g_err_flat)))
        g_rel_mean_all = float(
            np.mean(np.abs(g_err_flat) / (np.abs(g_true_flat) + 1e-12))
        )

        g_true_final = g_true_all[-1, :]
        g_pred_final = g_pred_all[-1, :]
        g_err_final  = g_pred_final - g_true_final

        g_L2_final  = float(np.linalg.norm(g_err_final))
        g_rel_final = float(
            g_L2_final / (np.linalg.norm(g_true_final) + 1e-12)
        )

        print(f"  Growth L2 (all times, masked)   = {g_L2_all:.4e}")
        print(f"  Growth rel (all times, masked)  = {g_rel_all:.4e}")
        print(f"  Growth mean |err| (all, masked) = {g_mean_abs_all:.4e}")
        print(f"  Growth mean rel (all, masked)   = {g_rel_mean_all:.4e}")
        print(f"  Growth L2 (final, masked)       = {g_L2_final:.4e}")
        print(f"  Growth rel (final, masked)      = {g_rel_final:.4e}")

        # ---------- Net area gain in cm^2 (total area increase) ----------
        # True nodal λ at final time (no mask)
        lamgx_true_final_nodal = lamgx_true[-1, :]   # (N_bottom,)
        lamgy_true_final_nodal = lamgy_true[-1, :]

        # Approximate true element λ by averaging nodal values over each element
        lamgx_true_elem = np.zeros(N_elem, dtype=np.float64)
        lamgy_true_elem = np.zeros(N_elem, dtype=np.float64)
        for e in range(N_elem):
            nodes_e = bottom_face_conn_local[e]
            lamgx_true_elem[e] = np.mean(lamgx_true_final_nodal[nodes_e])
            lamgy_true_elem[e] = np.mean(lamgy_true_final_nodal[nodes_e])

        # Predicted element λ at final time is directly from lamdag_eff
        lamgx_pred_elem = lamdag_eff[T_eff - 1, :, 0]   # (N_elem,)
        lamgy_pred_elem = lamdag_eff[T_eff - 1, :, 1]

        # ---- Scale by undeformed element area = 0.25 cm^2 ----
        AREA0_PER_ELEM_CM2 = 0.25  # cm^2 (since 25 mm^2 = 0.25 cm^2)

        # Total current areas in cm^2
        area_true = float(np.sum(lamgx_true_elem * lamgy_true_elem * AREA0_PER_ELEM_CM2))
        area_pred = float(np.sum(lamgx_pred_elem * lamgy_pred_elem * AREA0_PER_ELEM_CM2))

        # Initial area A0 in cm^2
        A0 = float(N_elem * AREA0_PER_ELEM_CM2)

        # Net area gain (cm^2); zero if no growth
        area_gain_true = area_true - A0
        area_gain_pred = area_pred - A0

        area_gain_err_abs = float(abs(area_gain_pred - area_gain_true))
        area_gain_err_rel = float(
            area_gain_err_abs / (abs(area_gain_true) + 1e-12)
        )

        print(f"  Net area gain (true, cm^2)      = {area_gain_true:.4e}")
        print(f"  Net area gain (pred, cm^2)      = {area_gain_pred:.4e}")
        print(f"  Net area gain abs error (cm^2)  = {area_gain_err_abs:.4e}")
        print(f"  Net area gain rel error         = {area_gain_err_rel:.4e}")

        # ---- Optional plotting of growth fields at final time (NO MASK) ----
        if plot_growth_fields and n_plotted < max_plot_sims:
            n_plotted += 1

            coords = bottom_xy
            x = coords[:, 0]
            y = coords[:, 1]

            gx_true_final_full = lamgx_true[-1, :]
            gy_true_final_full = lamgy_true[-1, :]
            gx_pred_final_full = lamgx_pred_nodal[-1, :]
            gy_pred_final_full = lamgy_pred_nodal[-1, :]

            gx_err_full = gx_pred_final_full - gx_true_final_full
            gy_err_full = gy_pred_final_full - gy_true_final_full

            # λ_gx
            fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
            titles_gx = [
                r"$\lambda_{gx}$ (true)",
                r"$\lambda_{gx}$ (pred)",
                r"error: pred - true",
            ]
            fields_gx = [gx_true_final_full, gx_pred_final_full, gx_err_full]
            for ax, field, title in zip(axes1, fields_gx, titles_gx):
                sc = ax.scatter(x, y, c=field, s=8, cmap="viridis")
                ax.set_aspect("equal")
                ax.set_xlabel("x")
                ax.set_ylabel("y")
                ax.set_title(f"Sim {sim_id} – {title}")
                fig1.colorbar(sc, ax=ax, shrink=0.8)

            # λ_gy
            fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
            titles_gy = [
                r"$\lambda_{gy}$ (true)",
                r"$\lambda_{gy}$ (pred)",
                r"error: pred - true",
            ]
            fields_gy = [gy_true_final_full, gy_pred_final_full, gy_err_full]
            for ax, field, title in zip(axes2, fields_gy, titles_gy):
                sc = ax.scatter(x, y, c=field, s=8, cmap="viridis")
                ax.set_aspect("equal")
                ax.set_xlabel("x")
                ax.set_ylabel("y")
                ax.set_title(f"Sim {sim_id} – {title}")
                fig2.colorbar(sc, ax=ax, shrink=0.8)

            plt.show()

        # ---- Store metrics for this sim ----
        all_growth_metrics.append({
            "sim_id": sim_id,
            "n_mask": n_mask,
            "g_L2_all": g_L2_all,
            "g_rel_all": g_rel_all,
            "g_mean_abs_all": g_mean_abs_all,
            "g_rel_mean_all": g_rel_mean_all,
            "g_L2_final": g_L2_final,
            "g_rel_final": g_rel_final,
            "area_gain_true": area_gain_true,
            "area_gain_pred": area_gain_pred,
            "area_gain_err_abs": area_gain_err_abs,
            "area_gain_err_rel": area_gain_err_rel,
        })

    return all_growth_metrics


In [ ]:
r = 9

growth_metrics_r9 = eval_growth_metrics_for_val_sims_from_zarr(
    val_sims=val_sims,              # your list of sim IDs
    r_modes=r,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
    plot_growth_fields=True,        # <– turn on plotting
    max_plot_sims=50,                # <– e.g., first 6 sims will get plots
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

def plot_growth_metrics_histograms(all_growth_metrics,
                                   r_modes=None,
                                   bins=20,
                                   save_stats_path=None):
    """
    Given all_growth_metrics (list of dicts from eval_growth_metrics_for_val_sims_from_zarr),
    make histograms of key growth metrics across all sims and compute summary statistics.

    Parameters
    ----------
    all_growth_metrics : list[dict]
        Output of eval_growth_metrics_for_val_sims_from_zarr.
    r_modes : int or None
        POD dimension (for labeling only).
    bins : int
        Number of histogram bins.
    save_stats_path : str or None
        If not None, save the summary statistics as JSON to this path.

    Returns
    -------
    stats : dict
        Nested dictionary of summary statistics for each metric.
    """

    # Collect each metric as numpy arrays
    sim_ids           = np.array([m["sim_id"]           for m in all_growth_metrics])
    n_mask            = np.array([m["n_mask"]           for m in all_growth_metrics])

    g_L2_all          = np.array([m["g_L2_all"]         for m in all_growth_metrics])
    g_rel_all         = np.array([m["g_rel_all"]        for m in all_growth_metrics])
    g_mean_abs_all    = np.array([m["g_mean_abs_all"]   for m in all_growth_metrics])
    g_rel_mean_all    = np.array([m["g_rel_mean_all"]   for m in all_growth_metrics])

    g_L2_final        = np.array([m["g_L2_final"]       for m in all_growth_metrics])
    g_rel_final       = np.array([m["g_rel_final"]      for m in all_growth_metrics])

    area_gain_true    = np.array([m["area_gain_true"]   for m in all_growth_metrics])
    area_gain_pred    = np.array([m["area_gain_pred"]   for m in all_growth_metrics])
    area_gain_err_abs = np.array([m["area_gain_err_abs"] for m in all_growth_metrics])
    area_gain_err_rel = np.array([m["area_gain_err_rel"] for m in all_growth_metrics])

    # ---- summary statistics helper ----
    def compute_stats(arr):
        return {
            "mean":   float(arr.mean()),
            "median": float(np.median(arr)),
            "std":    float(arr.std()),
            "min":    float(arr.min()),
            "max":    float(arr.max()),
            "p90":    float(np.percentile(arr, 90)),
            "p95":    float(np.percentile(arr, 95)),
        }

    stats = {
        "r_modes": int(r_modes) if r_modes is not None else None,
        "N": int(len(sim_ids)),
        "sim_ids": sim_ids.tolist(),
        "metrics": {
            "g_L2_all":          compute_stats(g_L2_all),
            "g_rel_all":         compute_stats(g_rel_all),
            "g_mean_abs_all":    compute_stats(g_mean_abs_all),
            "g_rel_mean_all":    compute_stats(g_rel_mean_all),
            "g_L2_final":        compute_stats(g_L2_final),
            "g_rel_final":       compute_stats(g_rel_final),
            "area_gain_true":    compute_stats(area_gain_true),
            "area_gain_pred":    compute_stats(area_gain_pred),
            "area_gain_err_abs": compute_stats(area_gain_err_abs),
            "area_gain_err_rel": compute_stats(area_gain_err_rel),
            "n_mask":            compute_stats(n_mask),
        },
    }

    # ---- print concise summary (focus on relative quantities) ----
    print("\n=== Growth summary over validation sims ===")
    if r_modes is not None:
        print(f"r = {r_modes}, N_sims = {len(sim_ids)}")
    else:
        print(f"N_sims = {len(sim_ids)}")
    print(f"Sims: {sim_ids}")

    def print_stats(name, s):
        print(
            f"{name:20s}: "
            f"mean={s['mean']:.4e}, median={s['median']:.4e}, "
            f"std={s['std']:.4e}, min={s['min']:.4e}, max={s['max']:.4e}, "
            f"p90={s['p90']:.4e}, p95={s['p95']:.4e}"
        )

    print_stats("g_rel_all",         stats["metrics"]["g_rel_all"])
    print_stats("g_rel_final",       stats["metrics"]["g_rel_final"])
    print_stats("g_rel_mean_all",    stats["metrics"]["g_rel_mean_all"])
    print_stats("area_gain_err_rel", stats["metrics"]["area_gain_err_rel"])
    print_stats("area_gain_true",    stats["metrics"]["area_gain_true"])
    print_stats("area_gain_pred",    stats["metrics"]["area_gain_pred"])

    # ---- histogram helper ----
    def _plot_hist(values, title, xlabel, logx=False):
        plt.figure(figsize=(6, 4))
        if logx:
            values = np.clip(values, a_min=1e-16, a_max=None)
            plt.hist(np.log10(values), bins=bins, alpha=0.8)
            plt.xlabel(f"log10({xlabel})")
        else:
            plt.hist(values, bins=bins, alpha=0.8)
            plt.xlabel(xlabel)

        if r_modes is not None:
            full_title = f"{title} (r={r_modes}, N={len(values)})"
        else:
            full_title = f"{title} (N={len(values)})"

        plt.title(full_title)
        plt.ylabel("Count")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ---- Make histograms ----
    _plot_hist(g_rel_all,      "Growth relative error (all times, masked)", "g_rel_all",      logx=False)
    _plot_hist(g_rel_final,    "Growth relative error (final, masked)",     "g_rel_final",    logx=False)
    _plot_hist(g_rel_mean_all, "Growth mean relative error (all, masked)",  "g_rel_mean_all", logx=False)

    _plot_hist(g_L2_all,   "Growth L2 error (all times, masked)", "g_L2_all",   logx=True)
    _plot_hist(g_L2_final, "Growth L2 error (final, masked)",     "g_L2_final", logx=True)
    _plot_hist(g_mean_abs_all, "Growth mean |err| (all, masked)", "g_mean_abs_all", logx=True)

    _plot_hist(area_gain_true,    "Net area gain (true)",          "area_gain_true",    logx=False)
    _plot_hist(area_gain_pred,    "Net area gain (pred)",          "area_gain_pred",    logx=False)
    _plot_hist(area_gain_err_abs, "Net area gain abs error",       "area_gain_err_abs", logx=True)
    # Net area gain relative error – clip for plotting only to avoid huge outliers
    area_gain_err_rel_plot = np.clip(area_gain_err_rel, 0.0, 4.0)
    n_clipped = np.sum(area_gain_err_rel > 4.0)
    if n_clipped > 0:
        print(f"[plot] area_gain_err_rel: {n_clipped} values > 2.0 were clipped for visualization.")

    plt.figure(figsize=(6, 4))
    plt.hist(area_gain_err_rel_plot, bins=bins, alpha=0.8)
    plt.xlabel("area_gain_err_rel")

    if r_modes is not None:
        title = f"Net area gain relative error (r={r_modes}, N={len(area_gain_err_rel)})"
    else:
        title = f"Net area gain relative error (N={len(area_gain_err_rel)})"

    plt.title(title)
    plt.ylabel("Count")
    plt.xlim(0.0, 4.0)   # Fix the x-axis bounds
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    _plot_hist(n_mask,            "Number of masked nodes",        "n_mask",            logx=False)

    # ---- optional: save stats to disk ----
    if save_stats_path is not None:
        with open(save_stats_path, "w") as f:
            json.dump(stats, f, indent=2)
        print(f"\nSaved growth stats to {save_stats_path}")

    return stats


In [ ]:
r = 9
growth_stats_r9 = plot_growth_metrics_histograms(
    all_growth_metrics=growth_metrics_r9,
    r_modes=r,
    bins=30,
    save_stats_path=f"growth_stats_r{r}.json",
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_net_area_gain_relative_error(
    all_growth_metrics,
    xlim=None,
    ylim=None,
    title="Net area gain relative error vs true net area gain (cm²)"
):
    """
    Scatter plot of relative net area gain prediction error vs true net area gain,
    now using cm² units on the x-axis.

    Parameters
    ----------
    all_growth_metrics : list of dict
        Output of eval_growth_metrics_for_val_sims_from_zarr(), containing:
            - area_gain_true (cm²)
            - area_gain_err_abs (cm²)
            - area_gain_err_rel

    xlim : tuple or None
        Optional (xmin, xmax) limits for true net area gain.

    ylim : tuple or None
        Optional (ymin, ymax) for relative error.

    title : str
        Plot title.
    """

    true_vals = []
    rel_errs  = []
    sim_ids   = []

    for m in all_growth_metrics:
        sim_id = m["sim_id"]
        area_gain_true = float(m["area_gain_true"])     # already cm²
        area_gain_err_abs = float(m["area_gain_err_abs"])

        # Skip pathological cases
        if abs(area_gain_true) < 1e-12:
            print(f"[WARN] Sim {sim_id}: area_gain_true ~ 0, skipping.")
            continue

        rel_err = area_gain_err_abs / abs(area_gain_true)

        sim_ids.append(sim_id)
        true_vals.append(area_gain_true)
        rel_errs.append(rel_err)

    true_vals = np.array(true_vals)
    rel_errs  = np.array(rel_errs)

    # ---- Plot ----
    plt.figure(figsize=(8, 6))
    plt.scatter(true_vals, rel_errs, s=30, alpha=0.7, edgecolors="k")

    plt.xlabel("True net area gain (cm²)")   # updated label
    plt.ylabel("Relative abs error = |pred - true| / |true|")
    plt.title(title)

    if xlim is not None:
        plt.xlim(xlim)
    if ylim is not None:
        plt.ylim(ylim)

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# metrics = eval_growth_metrics_for_val_sims_from_zarr(val_sims, r_modes=9)
plot_net_area_gain_relative_error(
    growth_metrics_r9,
    xlim=None,          # or something like (0, 400.0)
    ylim=(0.0, 2.0),    # if you want to focus on “reasonable” errors
)


## Graphics for Sim 344

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_volume_true_vs_pred(sim_id=294, r_modes=9):
    """
    Plot true vs predicted expander volume vs time for a given simulation.
    """

    # ---------- 1) True FE volume vs time ----------
    (_, _, _, volume_snap,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No snapshots found for sim_id={sim_id}")

    t_true = time_vals[idx].astype(float)
    vol_true = volume_snap[idx].astype(float)

    order = np.argsort(t_true)
    t_true = t_true[order]
    vol_true = vol_true[order]

    # ---------- 2) NODE rollout ----------
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    # Time array
    if "t" in roll:
        t_pred = np.asarray(roll["t"], dtype=float)
    elif "t_roll" in roll:
        t_pred = np.asarray(roll["t_roll"], dtype=float)
    else:
        if "state_traj" in roll:
            n_steps = roll["state_traj"].shape[0]
        elif "Y_seq" in roll:
            n_steps = roll["Y_seq"].shape[0]
        else:
            raise KeyError("Could not infer prediction time array.")
        dt = float(roll.get("dt", 1.4))
        t_pred = np.arange(n_steps) * dt

    # Predicted volume
    if "vol_pred" in roll:
        vol_pred = np.asarray(roll["vol_pred"], dtype=float)
    elif "vol_seq" in roll:
        vol_pred = np.asarray(roll["vol_seq"], dtype=float)
    else:
        if "state_traj" in roll:
            state = np.asarray(roll["state_traj"], dtype=float)
        elif "Y_seq" in roll:
            state = np.asarray(roll["Y_seq"], dtype=float)
        else:
            raise KeyError("Could not find predicted volume in rollout.")
        vol_pred = state[:, -1]

    # ---------- 3) Plot ----------
    plt.figure(figsize=(7, 5))

    # TRUE (blue circles, thick)
    plt.plot(
        t_true, vol_true,
        color="blue",
        marker="o",
        markersize=8,
        linewidth=2.5,
        label="True FE volume"
    )

    # PRED (black X markers, dashed, thick)
    plt.plot(
        t_pred, vol_pred,
        color="red",
        marker="x",
        markersize=8,
        linewidth=2.5,
        linestyle="--",
        label="NODE predicted volume"
    )

    plt.xlabel("Time (days)", fontsize=12)
    plt.ylabel("Expander volume", fontsize=12)
    plt.title(f"Sim {sim_id}: Volume vs Time", fontsize=14)

    #plt.legend(fontsize=11)
    plt.grid(False)  # turn off grid

    plt.tight_layout()
    plt.savefig("Sim_344_Vol_vs_time.png", dpi=600, transparent=True)
    plt.show()


In [ ]:
# Call it for your visualization sim:
plot_volume_true_vs_pred(sim_id=344, r_modes=9)

In [ ]:
import numpy as np
import zarr
import matplotlib.pyplot as plt

def rollout_and_compare_displacements(
    sim_id=294,
    r_modes=9,
    plot_errors=True,
):
    """
    For a given sim_id:
      - run NODE rollout
      - reconstruct nodal displacements u_pred(t)
      - extract true FE displacement snapshots u_true(t)
      - compute L2 and relative L2 errors vs time

    Returns a dict with:
      {
        "sim_id": sim_id,
        "t_true":        (T_eff,) array of FE times (days),
        "t_pred":        (T_eff,) array of rollout times (days),
        "u_true_seq":    (T_eff, N_nodes, 3) true displacements,
        "u_pred_seq":    (T_eff, N_nodes, 3) predicted displacements,
        "L2_err":        (T_eff,) absolute L2 error ||u_pred - u_true||_2,
        "rel_L2_err":    (T_eff,) relative L2 error,
      }
    """

    # ---------- 1) Load mapping + disp snapshots ----------
    g_disp = zarr.open_group(ZARR_DISP, mode="r")
    disp_snaps = np.asarray(g_disp[DISP_SNAP_DATASET], dtype=np.float64)
    # disp_snaps shape: (3*N_nodes, S) or similar flattened block

    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    # indices of FE snapshots for this sim
    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No FE snapshots found for sim_id={sim_id}")

    t_true_all = time_vals[idx].astype(float)
    order = np.argsort(t_true_all)
    idx_sorted = idx[order]
    t_true_all = t_true_all[order]

    print(f"[sim {sim_id}] FE frames: {len(t_true_all)}, "
          f"t_true[0]={t_true_all[0]:.3f}, t_true[-1]={t_true_all[-1]:.3f}")

    # ---------- 2) NODE rollout + reconstruction ----------
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    # reconstruct nodal displacements from rollout
    t_pred_all, u_nodes_seq_all = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )
    # u_nodes_seq_all: (T_roll, N_nodes, 3)

    print(f"[sim {sim_id}] NODE rollout steps: {len(t_pred_all)}, "
          f"t_pred[0]={t_pred_all[0]:.3f}, t_pred[-1]={t_pred_all[-1]:.3f}")

    # ---------- 3) Align trajectories (truncate to common length) ----------
    T_eff = min(len(t_true_all), len(t_pred_all))
    if T_eff < len(t_true_all) or T_eff < len(t_pred_all):
        print(f"  [WARN] Truncating time series from "
              f"{len(t_true_all)} (true), {len(t_pred_all)} (pred) to {T_eff} steps.")

    t_true = t_true_all[:T_eff]
    t_pred = t_pred_all[:T_eff]

    # truncate predicted displacements
    u_pred_seq = u_nodes_seq_all[:T_eff, :, :]  # (T_eff, N_nodes, 3)
    T_eff, N_nodes, _ = u_pred_seq.shape

    # ---------- 4) Build true displacement sequence ----------
    u_true_seq = np.zeros_like(u_pred_seq)  # (T_eff, N_nodes, 3)

    for k in range(T_eff):
        snap_idx = idx_sorted[k]
        u_true_flat = disp_snaps[:, snap_idx]  # (3*N_nodes,)

        ux_t, uy_t, uz_t = split_disp_block(u_true_flat)  # each (N_nodes,)
        if ux_t.shape[0] != N_nodes:
            raise ValueError(
                f"Mismatch in node count at step {k}: "
                f"ux_t has {ux_t.shape[0]} nodes, but u_pred has {N_nodes}"
            )

        u_true_seq[k, :, 0] = ux_t
        u_true_seq[k, :, 1] = uy_t
        u_true_seq[k, :, 2] = uz_t

    # ---------- 5) Compute errors vs time ----------
    L2_err = np.zeros(T_eff, dtype=float)
    rel_L2_err = np.zeros(T_eff, dtype=float)

    for k in range(T_eff):
        diff = u_pred_seq[k] - u_true_seq[k]        # (N_nodes, 3)
        num = np.linalg.norm(diff.ravel())
        denom = np.linalg.norm(u_true_seq[k].ravel()) + 1e-12

        L2_err[k] = num
        rel_L2_err[k] = num / denom
        if k ==0:
            rel_L2_err[k] = 0

    print(f"  Final step errors:"
          f" L2={L2_err[-1]:.4e}, rel={rel_L2_err[-1]:.4e}")

    # ---------- 6) Optional plotting ----------
    if plot_errors:
        plt.figure(figsize=(7, 5))
        plt.plot(t_true, L2_err, marker="o", label=r"$\|u_{\mathrm{pred}}-u_{\mathrm{true}}\|_2$")
        plt.xlabel("Time (days)")
        plt.ylabel("L2 displacement error")
        plt.title(f"Sim {sim_id}: Displacement L2 error vs time (r={r_modes})")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(7, 5))
        plt.plot(t_true, rel_L2_err, marker="o", label="relative L2 error")
        plt.xlabel("Time (days)")
        plt.ylabel("Relative L2 error")
        plt.title(f"Sim {sim_id}: Relative displacement error vs time (r={r_modes})")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ---------- 7) Package result ----------
    result = {
        "sim_id": sim_id,
        "t_true": t_true,
        "t_pred": t_pred,
        "u_true_seq": u_true_seq,
        "u_pred_seq": u_pred_seq,
        "L2_err": L2_err,
        "rel_L2_err": rel_L2_err,
    }
    return result


In [ ]:
# Example call for your visualization sim:
disp_results_294 = rollout_and_compare_displacements(sim_id=344, r_modes=9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_latent_trajectories_for_sim(sim_id=294, r_modes=9):
    """
    Plot the latent trajectories of the first 3 displacement modes
    for a given simulation, with custom colors, aesthetics, and thicker lines.
    """

    # Load latent data
    U_lat, _, _, _, sim_index, time_vals, _, _, _, _ = load_raw_data(r_modes)

    # Identify snapshots belonging to this simulation
    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No snapshots found for sim_id={sim_id}")

    # Sort by time
    t_sim = time_vals[idx].astype(float)
    order = np.argsort(t_sim)
    idx_sorted = idx[order]
    t_sorted = t_sim[order]

    # Extract latent coefficients for this simulation
    U_lat_sim = U_lat[idx_sorted, :]      # shape (T, r_modes)

    print(f"[sim {sim_id}] latent trajectory shape: {U_lat_sim.shape} "
          f"(T={len(t_sorted)}, r_modes={r_modes})")

    # --- Plot settings ---
    colors = ["black", "turquoise", "darkgreen"]
    plt.figure(figsize=(4, 3.5))

    for j in range(3):
        plt.plot(
            t_sorted,
            U_lat_sim[:, j],
            marker="x",
            markersize=8,
            linewidth=2.5,
            color=colors[j],
            label=f"mode {j+1}"
        )

    plt.xlabel("Time (days)", fontsize=12)
    plt.ylabel("Latent ", fontsize=12)
    plt.title(f"Sim {sim_id}: Latent Trajectories (modes 1–3)", fontsize=14)

    #plt.legend(loc="best", fontsize=10)
    plt.grid(False)  # remove grid
    plt.tight_layout()
    plt.savefig("Sim344_modes1_to_3_latent_trajectories.png", dpi=600, transparent=True)
    plt.show()


In [ ]:
# Example usage for your visualization sim:
plot_latent_trajectories_for_sim(sim_id=344, r_modes=9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_latent_trajectories_for_sim_normalized(sim_id=294, r_modes=9):
    """
    Plot the latent trajectories of the first r_modes displacement modes
    for a given simulation, in *normalized* space.

    Normalization: for each mode j,
        U_lat_norm[:, j] = (U_lat[:, j] - mean_j) / std_j
    where mean/std are computed over *all* snapshots in the dataset.

    Assumes:
      load_raw_data(r_modes) returns:
        U_lat, Lx_lat, Ly_lat, volume_snap,
        sim_index, time_vals,
        ...
      with:
        - U_lat.shape = (S, r_modes)
        - sim_index.shape = (S,)
        - time_vals.shape = (S,)
    """

    # Unpack U_lat, sim_index, time_vals from your existing loader
    U_lat, _, _, _, sim_index, time_vals, _, _, _, _ = load_raw_data(r_modes)

    # Global mode-wise normalization over all snapshots
    U_mean = U_lat.mean(axis=0)              # (r_modes,)
    U_std  = U_lat.std(axis=0) + 1e-12       # avoid divide-by-zero
    U_lat_norm = (U_lat - U_mean) / U_std    # (S, r_modes)

    print(f"U_lat shape:       {U_lat.shape}")
    print(f"U_lat_norm shape:  {U_lat_norm.shape}")
    print(f"sim_index shape:   {sim_index.shape}")
    print(f"time_vals shape:   {time_vals.shape}")

    # Indices of snapshots belonging to this simulation
    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No snapshots found for sim_id={sim_id}")

    # Sort by time
    t_sim = time_vals[idx].astype(float)
    order = np.argsort(t_sim)
    idx_sorted = idx[order]
    t_sorted = t_sim[order]

    # Extract normalized latent trajectories for this sim
    U_lat_sim_norm = U_lat_norm[idx_sorted, :]  # (T, r_modes)
    T = U_lat_sim_norm.shape[0]

    print(f"[sim {sim_id}] normalized latent trajectory shape: "
          f"{U_lat_sim_norm.shape} (T={T}, r_modes={r_modes})")

    # --- Plot normalized latent trajectories ---
    plt.figure(figsize=(8, 5))
    for j in range(r_modes):
        plt.plot(t_sorted,
                 U_lat_sim_norm[:, j],
                 marker="o",
                 lw=1.5,
                 label=f"mode {j+1}")

    plt.xlabel("Time (days)")
    plt.ylabel("Normalized latent coefficient")
    plt.title(f"Sim {sim_id}: normalized latent trajectories (first {r_modes} modes)")
    plt.grid(alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_latent_trajectories_for_sim_normalized(sim_id=344, r_modes=9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

def plot_deformed_geometry_over_time_for_sim_single(
    sim_id=294,
    r_modes=9,
    n_frames=4,
    cmap="viridis"
):
    """
    Plot deformed geometry for selected timepoints, generating
    ONE LARGE FIGURE PER FRAME.

    - Axes turned off (no grid, no panes)
    - Colorbars share the SAME scale across all frames
    - Timepoints equally spaced from start to end
    """

    # ---- 1) Rollout + reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    t, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )
    T, N_bottom, _ = u_nodes_seq.shape
    print(f"[sim {sim_id}] Loaded displacement seq: {u_nodes_seq.shape}")

    # ---- 2) Select timepoints ----
    idx_plot = np.linspace(0, T - 1, n_frames, dtype=int)
    t_plot = t[idx_plot]

    print("Frames to plot:")
    for i, idx_k in enumerate(idx_plot):
        print(f"  Frame {idx_k}: t = {t_plot[i]:.2f} days")

    # ---- 3) Reference coordinates ----
    x_ref = node_coords_bottom[:, 0]
    y_ref = node_coords_bottom[:, 1]
    z_ref = node_coords_bottom[:, 2]

    # ---- 4) Determine global color scale (max |uz| across all frames) ----
    max_abs_uz = 0.0
    for k in idx_plot:
        uz_k = u_nodes_seq[k, :, 2]
        max_abs_uz = max(max_abs_uz, np.max(np.abs(uz_k)))

    vmin = 0
    vmax =  30
    print(f"Color scale: vmin={vmin:.4f}, vmax={vmax:.4f}")

    # ---- 5) Generate ONE large figure per timepoint ----
    for i, k in enumerate(idx_plot):
        u_k = u_nodes_seq[k]
        ux_k, uy_k, uz_k = u_k[:, 0], u_k[:, 1], u_k[:, 2]

        x_def = x_ref + ux_k
        y_def = y_ref + uy_k
        z_def = z_ref + uz_k

        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection="3d")

        # Plot colored by uz
        sc = ax.scatter(
            x_def, y_def, z_def,
            c=uz_k,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=10
        )
         # ---- manually enforce color limits on the mappable ----
        sc.set_clim(vmin, vmax)

        # ----- Clean "no-grid" aesthetic -----
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.grid(False)

        # Remove panes
        ax.xaxis.pane.set_visible(False)
        ax.yaxis.pane.set_visible(False)
        ax.zaxis.pane.set_visible(False)

        # Remove axis lines
        ax.w_xaxis.line.set_color((1,1,1,0))
        ax.w_yaxis.line.set_color((1,1,1,0))
        ax.w_zaxis.line.set_color((1,1,1,0))

        # Equal aspect ratio
        ax.set_box_aspect((np.ptp(x_def), np.ptp(y_def), np.ptp(z_def)))

        # View angle
        ax.view_init(elev=30, azim=-60)

        # Title
        ax.set_title(f"Sim {sim_id} — t = {t_plot[i]:.2f} days", fontsize=16)

        # Consistent colorbar
        
        cbar = plt.colorbar(sc, ax=ax,shrink=0.75)
        cbar.set_label("Out-of-plane displacement $u_z$ (mm)", fontsize=14)
        cbar.ax.tick_params(labelsize=12)

        plt.tight_layout()
        plt.show()


In [ ]:
plot_deformed_geometry_over_time_for_sim(sim_id=344, r_modes=9, n_frames=3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

def plot_deformed_geometry_single_frame(
    sim_id=294,
    r_modes=9,
    k=None,
    vmin=0.0,
    vmax=30.0,
    cmap="viridis",
    save_path=None,
    dpi=600
):
    """
    Plot ONE frame of the deformed geometry with:
    - transparent background if saving
    - high-resolution output
    - unified color scale (vmin, vmax)

    Parameters
    ----------
    save_path : str or None
        If provided, saves figure to this path (png recommended).
    dpi : int
        Resolution for saving (default: 600).
    """

    # ---- Rollout + reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    t, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )
    T, N_bottom, _ = u_nodes_seq.shape

    if k is None:
        raise ValueError("Must specify a time index k.")
    if k < 0 or k >= T:
        raise ValueError(f"k={k} out of range (0 to {T-1}).")

    print(f"[sim {sim_id}] Plotting frame k={k}, t={t[k]:.2f} days")

    # ---- Reference node positions ----
    x_ref = node_coords_bottom[:, 0]
    y_ref = node_coords_bottom[:, 1]
    z_ref = node_coords_bottom[:, 2]

    # ---- Displacements ----
    u_k = u_nodes_seq[k]
    ux_k, uy_k, uz_k = u_k[:, 0], u_k[:, 1], u_k[:, 2]

    x_def = x_ref + ux_k
    y_def = y_ref + uy_k
    z_def = z_ref + uz_k

    # ---- Create figure ----
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        x_def, y_def, z_def,
        c=uz_k,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=12,
        alpha=1.0
    )
    sc.set_clim(vmin, vmax)

    # Hide axes for clean visualization
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)

    # No background panes
    ax.xaxis.pane.set_visible(False)
    ax.yaxis.pane.set_visible(False)
    ax.zaxis.pane.set_visible(False)

    # Hide axis lines (new Matplotlib API)
    ax.xaxis.line.set_color((1,1,1,0))
    ax.yaxis.line.set_color((1,1,1,0))
    ax.zaxis.line.set_color((1,1,1,0))

    # Aspect ratio
    ax.set_box_aspect((np.ptp(x_def), np.ptp(y_def), np.ptp(z_def)))

    # Camera
    ax.view_init(elev=30, azim=-60)

    # Title
    ax.set_title(f"Sim {sim_id} — t = {t[k]:.2f} days", fontsize=18)

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, shrink=0.78)
    cbar.set_label(r"$u_z$ (mm)", fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    # ---- Save figure ----
    if save_path is not None:
        print(f"Saving high-resolution figure to: {save_path}")
        fig.savefig(
            save_path,
            dpi=dpi,
            transparent=True,   # removes white background!
            bbox_inches="tight",
            pad_inches=0.05
        )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_deformed_geometry_single_frame(
    sim_id=344,
    r_modes=9,
    k=0,
    vmin=0,
    vmax=60,
    save_path="sim344_day0.png"
)

In [ ]:
plot_deformed_geometry_single_frame(
    sim_id=344,
    r_modes=9,
    k=68,
    vmin=0,
    vmax=50,
    save_path="sim344_day98.png"
)

In [ ]:
plot_deformed_geometry_single_frame(
    sim_id=344,
    r_modes=9,
    k=34,
    vmin=0,
    vmax=50,
    save_path="sim344_day47_6.png"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

def plot_uz_error_on_geometry_single_frame(
    sim_id=294,
    r_modes=9,
    k=None,
    vmin_err=None,
    vmax_err=None,
    cmap="coolwarm",
    save_path=None,
    dpi=600,
):
    """
    Plot the u_z error field (pred - true) on the predicted deformed geometry
    for a single time frame of a given simulation.

    - Uses NODE rollout + reconstruction for predicted displacements.
    - Uses FE snapshots for the true displacements.
    - Error field: u_z_pred - u_z_true at the same time index.
    - Geometry: predicted deformed bottom surface.

    Parameters
    ----------
    sim_id : int
        Simulation ID (e.g., 294).

    r_modes : int
        Number of displacement modes used in the NODE / POD.

    k : int or None
        Time index into the rollout. If None, uses the final frame.

    vmin_err, vmax_err : float or None
        Color scale limits for the error field. If both are None,
        they are chosen symmetrically from the data (±max|error|).

    cmap : str
        Matplotlib colormap for the error field.

    save_path : str or None
        If provided, saves the figure at high resolution with transparent background.

    dpi : int
        Figure resolution when saving.
    """

    # ---- 1) NODE rollout + reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    t_pred, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )  # u_nodes_seq: (T, N_bottom, 3)
    T, N_bottom, _ = u_nodes_seq.shape

    if k is None:
        k = T - 1  # default: final frame

    if k < 0 or k >= T:
        raise ValueError(f"k={k} out of range (0 to {T-1}).")

    print(f"[sim {sim_id}] Error plot for frame k={k}, t_pred={t_pred[k]:.2f} days")

    # ---- 2) Load FE snapshots and mapping to get true u_z for this sim/time ----
    # Assumes these globals / functions are available in your script:
    #   - ZARR_DISP, DISP_SNAP_DATASET
    #   - bottom_xy, bottom_face_conn_local
    #   - load_raw_data(r_modes)
    #   - split_disp_block
    import zarr
    g_disp = zarr.open_group(ZARR_DISP, mode="r")
    disp_snaps = np.asarray(g_disp[DISP_SNAP_DATASET], dtype=np.float64)

    # Mapping arrays from your raw data loader
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    # Find all FE frames for this sim and sort by time
    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No FE snapshots found for sim_id={sim_id}")

    t_true_all = time_vals[idx].astype(float)
    order = np.argsort(t_true_all)
    idx_sorted = idx[order]
    t_true_sorted = t_true_all[order]

    # Make sure we don't exceed the available FE frames
    T_eff = min(T, len(idx_sorted))
    if k >= T_eff:
        raise ValueError(
            f"k={k} exceeds available matched frames T_eff={T_eff}; "
            f"reduce k or check rollout length vs FE frames."
        )

    snap_idx_k = idx_sorted[k]
    print(f"  Matching to FE frame index {snap_idx_k}, t_true={t_true_sorted[k]:.2f} days")

    # True FE displacements (bottom nodes) at that snapshot
    u_true_flat = disp_snaps[:, snap_idx_k]         # (3 * N_bottom,)
    ux_true, uy_true, uz_true = split_disp_block(u_true_flat)
    assert uz_true.shape[0] == N_bottom

    # ---- 3) Predicted displacements at frame k ----
    u_pred_k = u_nodes_seq[k]                       # (N_bottom, 3)
    ux_pred = u_pred_k[:, 0]
    uy_pred = u_pred_k[:, 1]
    uz_pred = u_pred_k[:, 2]

    # ---- 4) Error in u_z ----
    uz_err = uz_pred - uz_true                      # (N_bottom,)

    # ---- 5) Geometry: predicted deformed bottom surface ----
    x_ref = node_coords_bottom[:, 0]
    y_ref = node_coords_bottom[:, 1]
    z_ref = node_coords_bottom[:, 2]

    x_def = x_ref + ux_pred
    y_def = y_ref + uy_pred
    z_def = z_ref + uz_pred

    # ---- 6) Color scale for error ----
    if vmin_err is None or vmax_err is None:
        max_abs_err = np.max(np.abs(uz_err))
        if max_abs_err < 1e-12:
            max_abs_err = 1e-12
        vmax_err_auto = max_abs_err
        vmin_err_auto = -max_abs_err
        if vmin_err is None:
            vmin_err = vmin_err_auto
        if vmax_err is None:
            vmax_err = vmax_err_auto

    print(f"  Error color scale: vmin={vmin_err:.4e}, vmax={vmax_err:.4e}")

    # ---- 7) Figure & plot ----
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        x_def, y_def, z_def,
        c=uz_err,
        cmap=cmap,
        vmin=vmin_err,
        vmax=vmax_err,
        s=12,
        alpha=1.0,
    )

    # Clean aesthetics: no ticks, no grid, invisible panes/axes
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)

    ax.xaxis.pane.set_visible(False)
    ax.yaxis.pane.set_visible(False)
    ax.zaxis.pane.set_visible(False)

    ax.xaxis.line.set_color((1, 1, 1, 0))
    ax.yaxis.line.set_color((1, 1, 1, 0))
    ax.zaxis.line.set_color((1, 1, 1, 0))

    # Aspect ratio
    ax.set_box_aspect((np.ptp(x_def), np.ptp(y_def), np.ptp(z_def)))

    # View angle
    ax.view_init(elev=30, azim=-60)

    # Title
    ax.set_title(
        f"Sim {sim_id} — $u_z$ error (pred - true), t = {t_pred[k]:.2f} days",
        fontsize=16,
    )

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, shrink=0.78)
    cbar.set_label(r"$u_z$ error (mm)", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    # ---- 8) Save figure (high-res, transparent) ----
    if save_path is not None:
        print(f"Saving high-resolution error figure to: {save_path}")
        fig.savefig(
            save_path,
            dpi=dpi,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.05,
        )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_uz_error_on_geometry_single_frame(
    sim_id=344,
    r_modes=9,
    k=68,                        # or k=T-1, but this will raise; better to query T first.
    vmin_err=0,
    vmax_err=3.0,
    cmap="jet",
    save_path="sim344_uz_error_final.png",
    dpi=600,
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
import zarr

def plot_lamgx_field_on_geometry_single_frame(
    sim_id=294,
    r_modes=9,
    k=None,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
    vmin=None,
    vmax=None,
    cmap="viridis",
    save_path=None,
    dpi=600,
):
    """
    Plot ONE frame of the predicted λ_gx field on the predicted deformed
    bottom-surface geometry for a given simulation.

    - Geometry: from NODE rollout (reconstruct_u_nodes_sequence_from_rollout).
    - Field: λ_gx from lamdag_hist_pred[..., 0], mapped element → nodal.

    Parameters
    ----------
    sim_id : int
        Simulation ID.

    r_modes : int
        Number of displacement modes for NODE.

    k : int or None
        Time index. If None, uses the last common frame across rollout and
        lamdag_hist_pred.

    lamdag_zarr_path : str
        Zarr path containing lamdag histories (sim_<id>/lamdag_hist_pred).

    vmin, vmax : float or None
        Color scale limits. If both None, they are set from min/max of λ_gx.

    cmap : str
        Colormap for λ_gx field.

    save_path : str or None
        If provided, save figure to this path (PNG recommended).

    dpi : int
        Resolution when saving.
    """

    # ---- 1) Rollout + geometry reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    t_pred, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )  # (T_geom, N_bottom, 3)
    T_geom, N_bottom, _ = u_nodes_seq.shape

    # ---- 2) Load lamdag history ----
    g_lam = zarr.open(lamdag_zarr_path, mode="r")
    grp_name = f"sim_{sim_id}"
    if grp_name not in g_lam:
        raise KeyError(
            f"{grp_name} not found in {lamdag_zarr_path}. "
            "Did you run precompute_lamdag_for_val_sims first?"
        )

    g_sim = g_lam[grp_name]
    lamdag_hist_pred = np.asarray(g_sim["lamdag_hist_pred"][:], dtype=np.float64)
    # shape: (T_lam, N_elem, 2)
    T_lam, N_elem, _ = lamdag_hist_pred.shape

    # ---- 3) Align lengths between rollout and lamdag ----
    T_eff = min(T_geom, T_lam)
    if T_eff == 0:
        raise RuntimeError("No overlapping frames between rollout and lamdag histories.")

    if k is None:
        k = T_eff - 1
    if k < 0 or k >= T_eff:
        raise ValueError(f"k={k} out of valid range (0 to {T_eff-1}).")

    print(f"[sim {sim_id}] λ_gx field at frame k={k}, t={t_pred[k]:.2f} days")

    # ---- 4) Predicted λ_gx at element and nodal levels ----
    lamgx_elem_k = lamdag_hist_pred[k, :, 0]   # (N_elem,)

    lamgx_nodal = np.zeros(N_bottom, dtype=np.float64)
    counts = np.zeros(N_bottom, dtype=np.int32)

    for e in range(N_elem):
        lamgx_e = lamgx_elem_k[e]
        nodes_e = bottom_face_conn_local[e]
        for nid in nodes_e:
            lamgx_nodal[nid] += lamgx_e
            counts[nid] += 1

    nonzero = counts > 0
    lamgx_nodal[nonzero] /= counts[nonzero]
    zero = ~nonzero
    if np.any(zero):
        lamgx_nodal[zero] = 1.0  # neutral if no attached elements

    # ---- 5) Predicted deformed geometry at frame k ----
    x_ref = node_coords_bottom[:, 0]
    y_ref = node_coords_bottom[:, 1]
    z_ref = node_coords_bottom[:, 2]

    u_k = u_nodes_seq[k]
    ux_k, uy_k, uz_k = u_k[:, 0], u_k[:, 1], u_k[:, 2]

    x_def = x_ref + ux_k
    y_def = y_ref + uy_k
    z_def = z_ref + uz_k

    # ---- 6) Color scale for λ_gx ----
    if vmin is None or vmax is None:
        val_min = float(np.min(lamgx_nodal))
        val_max = float(np.max(lamgx_nodal))
        if vmin is None:
            vmin = val_min
        if vmax is None:
            vmax = val_max

    print(f"  λ_gx color scale: vmin={vmin:.4e}, vmax={vmax:.4e}")

    # ---- 7) Plot ----
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        x_def, y_def, z_def,
        c=lamgx_nodal,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=12,
        alpha=1.0,
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)

    ax.xaxis.pane.set_visible(False)
    ax.yaxis.pane.set_visible(False)
    ax.zaxis.pane.set_visible(False)

    ax.xaxis.line.set_color((1, 1, 1, 0))
    ax.yaxis.line.set_color((1, 1, 1, 0))
    ax.zaxis.line.set_color((1, 1, 1, 0))

    ax.set_box_aspect((np.ptp(x_def), np.ptp(y_def), np.ptp(z_def)))
    ax.view_init(elev=30, azim=-60)

    ax.set_title(
        fr"Sim {sim_id} — $\lambda_{{gx}}$ field, t = {t_pred[k]:.2f} days",
        fontsize=18,
    )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.78)
    cbar.set_label(r"$\lambda_{gx}$", fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    if save_path is not None:
        print(f"Saving λ_gx field figure to: {save_path}")
        fig.savefig(
            save_path,
            dpi=dpi,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.05,
        )

    plt.tight_layout()
    plt.show()


def plot_lamgy_field_on_geometry_single_frame(
    sim_id=294,
    r_modes=9,
    k=None,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
    vmin=None,
    vmax=None,
    cmap="viridis",
    save_path=None,
    dpi=600,
):
    """
    Plot ONE frame of the predicted λ_gy field on the predicted deformed
    bottom-surface geometry for a given simulation.

    - Geometry: from NODE rollout.
    - Field: λ_gy from lamdag_hist_pred[..., 1], mapped element → nodal.
    """

    # ---- 1) Rollout + geometry reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )

    t_pred, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )  # (T_geom, N_bottom, 3)
    T_geom, N_bottom, _ = u_nodes_seq.shape

    # ---- 2) Load lamdag history ----
    g_lam = zarr.open(lamdag_zarr_path, mode="r")
    grp_name = f"sim_{sim_id}"
    if grp_name not in g_lam:
        raise KeyError(
            f"{grp_name} not found in {lamdag_zarr_path}. "
            "Did you run precompute_lamdag_for_val_sims first?"
        )

    g_sim = g_lam[grp_name]
    lamdag_hist_pred = np.asarray(g_sim["lamdag_hist_pred"][:], dtype=np.float64)
    # shape: (T_lam, N_elem, 2)
    T_lam, N_elem, _ = lamdag_hist_pred.shape

    # ---- 3) Align lengths between rollout and lamdag ----
    T_eff = min(T_geom, T_lam)
    if T_eff == 0:
        raise RuntimeError("No overlapping frames between rollout and lamdag histories.")

    if k is None:
        k = T_eff - 1
    if k < 0 or k >= T_eff:
        raise ValueError(f"k={k} out of valid range (0 to {T_eff-1}).")

    print(f"[sim {sim_id}] λ_gy field at frame k={k}, t={t_pred[k]:.2f} days")

    # ---- 4) Predicted λ_gy at element and nodal levels ----
    lamgy_elem_k = lamdag_hist_pred[k, :, 1]   # (N_elem,)

    lamgy_nodal = np.zeros(N_bottom, dtype=np.float64)
    counts = np.zeros(N_bottom, dtype=np.int32)

    for e in range(N_elem):
        lamgy_e = lamgy_elem_k[e]
        nodes_e = bottom_face_conn_local[e]
        for nid in nodes_e:
            lamgy_nodal[nid] += lamgy_e
            counts[nid] += 1

    nonzero = counts > 0
    lamgy_nodal[nonzero] /= counts[nonzero]
    zero = ~nonzero
    if np.any(zero):
        lamgy_nodal[zero] = 1.0

    # ---- 5) Predicted deformed geometry at frame k ----
    x_ref = node_coords_bottom[:, 0]
    y_ref = node_coords_bottom[:, 1]
    z_ref = node_coords_bottom[:, 2]

    u_k = u_nodes_seq[k]
    ux_k, uy_k, uz_k = u_k[:, 0], u_k[:, 1], u_k[:, 2]

    x_def = x_ref + ux_k
    y_def = y_ref + uy_k
    z_def = z_ref + uz_k

    # ---- 6) Color scale for λ_gy ----
    if vmin is None or vmax is None:
        val_min = float(np.min(lamgy_nodal))
        val_max = float(np.max(lamgy_nodal))
        if vmin is None:
            vmin = val_min
        if vmax is None:
            vmax = val_max

    print(f"  λ_gy color scale: vmin={vmin:.4e}, vmax={vmax:.4e}")

    # ---- 7) Plot ----
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        x_def, y_def, z_def,
        c=lamgy_nodal,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=12,
        alpha=1.0,
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)

    ax.xaxis.pane.set_visible(False)
    ax.yaxis.pane.set_visible(False)
    ax.zaxis.pane.set_visible(False)

    ax.xaxis.line.set_color((1, 1, 1, 0))
    ax.yaxis.line.set_color((1, 1, 1, 0))
    ax.zaxis.line.set_color((1, 1, 1, 0))

    ax.set_box_aspect((np.ptp(x_def), np.ptp(y_def), np.ptp(z_def)))
    ax.view_init(elev=30, azim=-60)

    ax.set_title(
        fr"Sim {sim_id} — $\lambda_{{gy}}$ field, t = {t_pred[k]:.2f} days",
        fontsize=18,
    )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.78)
    cbar.set_label(r"$\lambda_{gy}$", fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    if save_path is not None:
        print(f"Saving λ_gy field figure to: {save_path}")
        fig.savefig(
            save_path,
            dpi=dpi,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.05,
        )

    plt.tight_layout()
    plt.show()


In [ ]:
# say you already know k values you like, e.g. k=0, 25, 50, 75
plot_lamgx_field_on_geometry_single_frame(sim_id=344, r_modes=9, k=68,
                                          vmin=1.0, vmax=1.4,
                                          save_path="sim344_lamgx_day98.png")


In [ ]:
# say you already know k values you like, e.g. k=0, 25, 50, 75
plot_lamgy_field_on_geometry_single_frame(sim_id=344, r_modes=9, k=68,
                                          vmin=1.0, vmax=1.6,
                                          save_path="sim344_lamgy_day98.png")

In [ ]:
print("t[0] =", t[0])
print("t[1] - t[0] =", t[1] - t[0])
print("t[k] =", t[30])
print("effective dt ≈", (t[-1] - t[0]) / (len(t) - 1))

In [ ]:
def debug_time_alignment(sim_id, r_modes=9):
    _, _, _, _, sim_index, time_vals, _, _, _, _ = load_raw_data(r_modes)

    idx = np.where(sim_index == sim_id)[0]
    t = time_vals[idx]
    order = np.argsort(t)
    t_sorted = t[order]

    print("Earliest FE snapshot time:", t_sorted[0])
    print("Latest FE snapshot time:",   t_sorted[-1])
    print("Number of FE snapshots:",    len(t_sorted))

    print("\nFirst few time differences:")
    print(np.diff(t_sorted[:10]))

    print("\nLast few time differences:")
    print(np.diff(t_sorted[-10:]))

debug_time_alignment(344)


In [ ]:
job_id, _ , _, _ = get_growth_params_for_sim(344)
print("Job corresponding to sim 344:", job_id)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_uz_topdown_single_frame(
    sim_id=294,
    r_modes=9,
    k=None,            # time index
    vmin=0.0,
    vmax=30.0,
    cmap="viridis",
    save_path=None,
    dpi=600
):
    """
    Plot u_z from a TOP-DOWN perspective (x-y plane).
    
    - Pure 2D scatter (top view)
    - Color = u_z displacement
    - Uses reconstructed u_nodes_seq from rollout
    """

    # ---- Rollout & reconstruction ----
    roll = rollout_single_sim(
        r_modes=r_modes,
        sim_id=sim_id,
        max_steps=None,
    )
    t, u_nodes_seq = reconstruct_u_nodes_sequence_from_rollout(
        roll=roll,
        r_modes=r_modes,
    )
    T, N_bottom, _ = u_nodes_seq.shape

    if k is None or k < 0 or k >= T:
        raise ValueError(f"Must choose k between 0 and {T-1}")

    print(f"[sim {sim_id}] Top-down u_z plot at k={k}, t={t[k]:.2f} days")

    # ---- Geometry ----
    x = node_coords_bottom[:, 0]
    y = node_coords_bottom[:, 1]
    uz = u_nodes_seq[k, :, 2]

    # ---- Create figure ----
    fig, ax = plt.subplots(figsize=(7, 6))

    sc = ax.scatter(
        x, y,
        c=uz,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=48
    )
    sc.set_clim(vmin, vmax)

    # Clean visualization
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Sim {sim_id} — Top-down $u_z$ at t={t[k]:.2f} days", fontsize=16)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(r"$u_z$ (mm)", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    plt.tight_layout()

    # ---- Save ----
    if save_path is not None:
        print(f"Saving figure to: {save_path}")
        fig.savefig(
            save_path,
            dpi=dpi,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.05,
        )

    plt.show()


In [ ]:
# Plot frame k=68 (≈98 days)
plot_uz_topdown_single_frame(
    sim_id=344,
    r_modes=9,
    k=68,
    vmin=0,
    vmax=50,
    save_path="sim344_topdown_uz_day98.png"
)
